# Dorogokupets17 Iron EoS

This section provides comprehensive testing and benchmarking of the `Dorogokupets17` class implementation.

**Contents:**
1. [Setup and Imports](#setup)
2. [Basic Functionality Tests](#basic-tests)
3. [Reference Condition Validation](#reference-validation)
4. [Convergence Tests](#convergence)
5. [Runtime Benchmarking](#runtime)
6. [Diagnostic Plots](#diagnostics)
7. [Thermodynamic Consistency Tests](#consistency)
8. [Edge Cases and Error Handling](#edge-cases)
9. [Comparison with Published Data](#comparison)

---

## Setup and Imports <a name="setup"></a>

Import necessary libraries and initialize the EoS instances.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

from iron_eos import Dorogokupets17

# Set plotting style
plt.style.use('tableau-colorblind10')

# Physical constants
GPa = 1e9  # Conversion factor

print("✓ Imports successful")

In [ ]:
# Initialize EoS instances for both phases
eos_bcc = Dorogokupets17(phase='bcc')
eos_fcc = Dorogokupets17(phase='fcc')

print("✓ EoS instances created")
print(f"  - bcc-Fe: V_0 = {eos_bcc.params['V0']*1e6:.4f} cm³/mol, K_0 = {eos_bcc.params['K0']/GPa:.1f} GPa")
print(f"  - fcc-Fe: V_0 = {eos_fcc.params['V0']*1e6:.4f} cm³/mol, K_0 = {eos_fcc.params['K0']/GPa:.1f} GPa")

---
## Basic Functionality Tests <a name="basic-tests"></a>

Test that all public methods execute without errors and return reasonable values.

In [ ]:
# Test conditions: room pressure and temperature
P_test = 1e5  # Pa (0.1 MPa)
T_test = 298.15  # K

print("Testing bcc-Fe at reference conditions (P = 0.1 MPa, T = 298.15 K):")
print("=" * 70)

methods = [
    ('density', 'kg/m³'),
    ('specific_internal_energy', 'J/kg'),
    ('specific_entropy', 'J/(kg·K)'),
    ('isobaric_heat_capacity', 'J/(kg·K)'),
    ('isochoric_heat_capacity', 'J/(kg·K)'),
    ('thermal_expansion', 'K⁻¹'),
    ('adiabatic_gradient', 'dimensionless')
]

results_bcc = {}
for method_name, units in methods:
    try:
        method = getattr(eos_bcc, method_name)
        result = method(P_test, T_test)
        results_bcc[method_name] = result
        print(f"  {method_name:30s}: {result:12.6e} {units}")
    except Exception as e:
        print(f"  {method_name:30s}: ERROR - {e}")
        results_bcc[method_name] = None

print("\n✓ All methods executed successfully" if all(v is not None for v in results_bcc.values()) 
      else "\n✗ Some methods failed")

In [ ]:
# Test fcc-Fe at elevated conditions
P_test_fcc = 10 * GPa  # 10 GPa
T_test_fcc = 1500  # K

print(f"Testing fcc-Fe at P = {P_test_fcc/GPa:.1f} GPa, T = {T_test_fcc:.0f} K:")
print("=" * 70)

results_fcc = {}
for method_name, units in methods:
    try:
        method = getattr(eos_fcc, method_name)
        result = method(P_test_fcc, T_test_fcc)
        results_fcc[method_name] = result
        print(f"  {method_name:30s}: {result:12.6e} {units}")
    except Exception as e:
        print(f"  {method_name:30s}: ERROR - {e}")
        results_fcc[method_name] = None

print("\n✓ All methods executed successfully" if all(v is not None for v in results_fcc.values()) 
      else "\n✗ Some methods failed")

---
## Reference Condition Validation <a name="reference-validation"></a>

Compare calculated properties at reference conditions with known values from the literature.

In [ ]:
# Reference values for bcc-Fe at 298.15 K and 0.1 MPa
# From Dorogokupets et al. (2017) and Desai (1986)
reference_bcc = {
    'density': 7874,  # kg/m³ (from V_0 = 7.092 cm³/mol)
    'isobaric_heat_capacity': 449,  # J/(kg·K) (approximately 25.1 J/(mol·K))
    'thermal_expansion': 3.55e-5,  # K⁻¹ (at 298 K)
}

P_ref = 1e5  # Pa
T_ref = 298.15  # K

print("Validation against reference values for bcc-Fe:")
print("=" * 80)
print(f"{'Property':30s} {'Calculated':>15s} {'Reference':>15s} {'Diff (%)':>12s}")
print("=" * 80)

for prop_name, ref_value in reference_bcc.items():
    method = getattr(eos_bcc, prop_name)
    calc_value = method(P_ref, T_ref)
    diff_percent = 100 * (calc_value - ref_value) / ref_value
    
    status = "✓" if abs(diff_percent) < 5 else "⚠"
    print(f"{prop_name:30s} {calc_value:15.6e} {ref_value:15.6e} {diff_percent:11.2f}% {status}")

---
## Convergence Tests <a name="convergence"></a>

Test the numerical convergence of the root-finding algorithm used to determine volume from pressure.

In [ ]:
# Test convergence by checking P(V(P,T), T) ≈ P
test_pressures = np.logspace(5, 11, 20)  # From 0.1 MPa to 100 GPa
test_temperatures = [300, 1000, 2000, 3000]  # K

print("Convergence test: P(V(P,T), T) - P should be ~ 0")
print("=" * 80)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=300)

for phase, eos, ax in [('bcc', eos_bcc, ax1), ('fcc', eos_fcc, ax2)]:
    max_errors = []
    
    for T in test_temperatures:
        errors = []
        for P in test_pressures:
            try:
                # Find volume for given P, T
                V = eos._find_volume(P, T)
                # Calculate pressure from that volume
                P_calc = eos._total_pressure(V, T)
                # Relative error
                rel_error = abs(P_calc - P) / P
                errors.append(rel_error)
            except:
                errors.append(np.nan)
        
        max_error = np.nanmax(errors) if len(errors) > 0 else np.nan
        max_errors.append(max_error)
        ax.loglog(test_pressures / GPa, errors, 'o-', label=f'$T = {T}$ K', alpha=0.7)
    
    ax.axhline(1e-5, color='k', linestyle='--', alpha=0.3, label='Target ($10^{-5}$)')
    ax.set_xlabel('Pressure [GPa]')
    ax.set_ylabel('Relative error in pressure')
    ax.set_title(f'{phase}-Fe convergence')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    print(f"{phase}-Fe: Max relative error = {np.nanmax(max_errors):.2e}")

plt.tight_layout()
plt.show()

---
## Runtime Benchmarking <a name="runtime"></a>

Measure the execution time of various methods to assess computational efficiency.

In [ ]:
# Single-call timing
n_iterations = 1000
P_bench = 50 * GPa
T_bench = 2000

print(f"Runtime benchmark: {n_iterations} calls at P = {P_bench/GPa:.0f} GPa, T = {T_bench:.0f} K")
print("=" * 80)

timing_results = {}

for phase, eos in [('bcc', eos_bcc), ('fcc', eos_fcc)]:
    print(f"\n{phase}-Fe:")
    timing_results[phase] = {}
    
    for method_name, units in methods:
        method = getattr(eos, method_name)
        
        # Warm-up call
        _ = method(P_bench, T_bench)
        
        # Timed calls
        start = time.perf_counter()
        for _ in range(n_iterations):
            _ = method(P_bench, T_bench)
        end = time.perf_counter()
        
        avg_time_ms = (end - start) / n_iterations * 1000
        timing_results[phase][method_name] = avg_time_ms
        print(f"  {method_name:30s}: {avg_time_ms:8.4f} ms/call")

print("\n✓ Runtime benchmark complete")

In [ ]:
# Visualize timing results
fig, ax = plt.subplots(figsize=(7, 4), dpi=300)

x = np.arange(len(methods))
width = 0.35

method_names = [m[0] for m in methods]
bcc_times = [timing_results['bcc'][m] for m in method_names]
fcc_times = [timing_results['fcc'][m] for m in method_names]

ax.bar(x - width/2, bcc_times, width, label='bcc-Fe', alpha=0.8)
ax.bar(x + width/2, fcc_times, width, label='fcc-Fe', alpha=0.8)

ax.set_ylabel('Time [ms/call]')
ax.set_xticks(x)
ax.set_xticklabels([m.replace('_', '\n') for m in method_names], rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Vectorized benchmark: measure time for array of conditions
n_points = 100
P_array = np.linspace(1e5, 100*GPa, n_points)
T_array = np.linspace(300, 3000, n_points)

print(f"Vectorized benchmark: {n_points} pressure-temperature points")
print("=" * 80)

for phase, eos in [('bcc', eos_bcc), ('fcc', eos_fcc)]:
    start = time.perf_counter()
    densities = np.array([eos.density(P, T) for P, T in zip(P_array, T_array)])
    end = time.perf_counter()
    
    total_time = end - start
    time_per_point = total_time / n_points * 1000
    
    print(f"{phase}-Fe: {total_time:.3f} s total ({time_per_point:.3f} ms/point)")

print("\n✓ Vectorized benchmark complete")

---
## Diagnostic Plots <a name="diagnostics"></a>

Create comprehensive diagnostic plots showing behavior across P-T space.

### Isotherms and Isobars

In [ ]:
# Density isotherms
pressures = np.linspace(0.1*GPa, 100*GPa, 100)
temperatures_iso = [300, 1000, 2000, 3000]

fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=300)

for phase, eos, col in [('bcc', eos_bcc, 0), ('fcc', eos_fcc, 1)]:
    
    # Density isotherms
    ax = axes[0, col]
    for T in temperatures_iso:
        densities = []
        for P in pressures:
            try:
                rho = eos.density(P, T)
                densities.append(rho)
            except:
                densities.append(np.nan)
        ax.plot(pressures/GPa, densities, label=f'{T} K', linewidth=2)
    
    ax.set_xlabel('Pressure [GPa]')
    ax.set_ylabel('Density [kg/m³]')
    ax.set_title(f'{phase}-Fe: Density isotherms')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Density isobars
    ax = axes[1, col]
    temps = np.linspace(300, 4000, 100)
    pressures_iso = [0.1*GPa, 1*GPa, 10*GPa, 50*GPa, 100*GPa]
    
    for P in pressures_iso:
        densities = []
        for T in temps:
            try:
                rho = eos.density(P, T)
                densities.append(rho)
            except:
                densities.append(np.nan)
        ax.plot(temps, densities, label=f'{P/GPa:.1f} GPa', linewidth=2)
    
    ax.set_xlabel('Temperature [K]')
    ax.set_ylabel('Density [kg/m³]')
    ax.set_title(f'{phase}-Fe: Density isobars')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Heat Capacity Temperature Dependence

In [ ]:
temps = np.linspace(300, 2000, 100)
pressures_cp = [0.1*GPa, 10*GPa, 50*GPa]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=300)

for phase, eos, ax in [('bcc', eos_bcc, axes[0]), ('fcc', eos_fcc, axes[1])]:
    for i, P in enumerate(pressures_cp):
        cp_values = []
        cv_values = []
        
        for T in temps:
            try:
                cp = eos.isobaric_heat_capacity(P, T)
                cv = eos.isochoric_heat_capacity(P, T)
                cp_values.append(cp)
                cv_values.append(cv)
            except:
                cp_values.append(np.nan)
                cv_values.append(np.nan)
        
        ax.plot(temps, cp_values, '-',  label=f'$C_P$ @ {P/GPa:.1f} GPa', color=f'C{i}')
        ax.plot(temps, cv_values, '--', label=f'$C_V$ @ {P/GPa:.1f} GPa', color=f'C{i}')
    
    ax.set_xlabel('Temperature [K]')
    ax.set_ylabel('Heat capacity [J/(kg·K)]')
    ax.set_title(f'{phase}-Fe')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Thermal Expansion and Adiabatic Gradient

In [ ]:
pressures_therm = np.linspace(1*GPa, 100*GPa, 50)
temperatures_therm = [500, 1000, 1500, 2000]

fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=300)

for phase, eos, col in [('bcc', eos_bcc, 0), ('fcc', eos_fcc, 1)]:
    
    # Thermal expansion vs pressure
    ax = axes[0, col]
    for T in temperatures_therm:
        alpha_values = []
        for P in pressures_therm:
            try:
                alpha = eos.thermal_expansion(P, T)
                alpha_values.append(alpha * 1e5)  # Convert to 10⁻⁵ K⁻¹
            except:
                alpha_values.append(np.nan)
        ax.plot(pressures_therm/GPa, alpha_values, label=f'{T} K', linewidth=2)
    
    ax.set_xlabel('Pressure [GPa]')
    ax.set_ylabel('Thermal expansion [$10^{-5}$ K$^{-1}$]')
    ax.set_title(f'{phase}-Fe')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Adiabatic gradient vs pressure
    ax = axes[1, col]
    for T in temperatures_therm:
        gamma_values = []
        for P in pressures_therm:
            try:
                gamma = eos.adiabatic_gradient(P, T)
                gamma_values.append(gamma)
            except:
                gamma_values.append(np.nan)
        ax.plot(pressures_therm/GPa, gamma_values, label=f'{T} K', linewidth=2)
    
    ax.set_xlabel('Pressure [GPa]')
    ax.set_ylabel('Adiabatic gradient')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### P-T Phase Space Heatmap

In [ ]:
# Create 2D grid in P-T space and calculate density
n_P = 50
n_T = 50
P_grid = np.linspace(1*GPa, 100*GPa, n_P)
T_grid = np.linspace(300, 2500, n_T)
P_mesh, T_mesh = np.meshgrid(P_grid, T_grid)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=300)

for phase, eos, ax in [('bcc', eos_bcc, axes[0]), ('fcc', eos_fcc, axes[1])]:
    density_grid = np.zeros_like(P_mesh)
    
    for i in range(n_T):
        for j in range(n_P):
            try:
                density_grid[i, j] = eos.density(P_mesh[i, j], T_mesh[i, j])
            except:
                density_grid[i, j] = np.nan
    
    im = ax.contourf(P_mesh/GPa, T_mesh, density_grid, levels=20, cmap='viridis')
    
    ax.set_xlabel('Pressure [GPa]')
    ax.set_ylabel('Temperature [K]')
    ax.set_title(f'{phase}-Fe')
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Density [kg/m³]')

plt.tight_layout()
plt.show()

---
## Thermodynamic Consistency Tests <a name="consistency"></a>

Verify thermodynamic relationships and Maxwell relations.

### Test: Cp ≥ Cv (always true for stable phases)

In [ ]:
print("Testing Cp ≥ Cv constraint:")
print("=" * 80)

test_conditions = [
    (1e5, 300),
    (10*GPa, 1000),
    (50*GPa, 1500),
    (100*GPa, 2000)
]

violations = 0
for phase, eos in [('bcc', eos_bcc), ('fcc', eos_fcc)]:
    print(f"\n{phase}-Fe:")
    for P, T in test_conditions:
        cp = eos.isobaric_heat_capacity(P, T)
        cv = eos.isochoric_heat_capacity(P, T)
        ratio = cp / cv
        
        status = "✓" if cp >= cv else "✗"
        if cp < cv:
            violations += 1
        
        print(f"  P={P/GPa:6.1f} GPa, T={T:5.0f} K: Cp/Cv = {ratio:.4f} {status}")

if violations == 0:
    print("\n✓ All tests passed: Cp ≥ Cv everywhere")
else:
    print(f"\n✗ Found {violations} violations of Cp ≥ Cv")

### Test: Entropy increases with temperature at constant pressure

In [ ]:
print("Testing dS/dT > 0 at constant pressure:")
print("=" * 80)

P_test_s = 50 * GPa
T_test_s = np.linspace(500, 3000, 50)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=300)

for phase, eos, ax in [('bcc', eos_bcc, axes[0]), ('fcc', eos_fcc, axes[1])]:
    S_values = []
    for T in T_test_s:
        S = eos.specific_entropy(P_test_s, T)
        S_values.append(S)
    
    # Check monotonicity
    dS = np.diff(S_values)
    all_positive = np.all(dS > 0)
    
    ax.plot(T_test_s, S_values, 'o-', linewidth=2)
    ax.set_xlabel('Temperature [K]')
    ax.set_ylabel('Specific entropy [J/(kg·K)]')
    ax.set_title(f'{phase}-Fe at {P_test_s/GPa:.0f} GPa')
    ax.grid(True, alpha=0.3)
    
    status = "✓" if all_positive else "✗"
    ax.text(0.05, 0.95, f'd$S$/d$T > 0$: {status}', transform=ax.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    print(f"{phase}-Fe: dS/dT > 0? {status}")

plt.tight_layout()
plt.show()

### Test: Density increases with pressure at constant temperature

In [ ]:
print("Testing dρ/dP > 0 at constant temperature:")
print("=" * 80)

T_test_rho = 1500  # K
P_test_rho = np.linspace(1*GPa, 100*GPa, 50)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=300)

for phase, eos, ax in [('bcc', eos_bcc, axes[0]), ('fcc', eos_fcc, axes[1])]:
    rho_values = []
    for P in P_test_rho:
        rho = eos.density(P, T_test_rho)
        rho_values.append(rho)
    
    # Check monotonicity
    drho = np.diff(rho_values)
    all_positive = np.all(drho > 0)
    
    ax.plot(P_test_rho/GPa, rho_values, 'o-', linewidth=2)
    ax.set_xlabel('Pressure [GPa]')
    ax.set_ylabel('Density [kg/m³]')
    ax.set_title(f'{phase}-Fe at {T_test_rho:.0f} K')
    ax.grid(True, alpha=0.3)
    
    status = "✓" if all_positive else "✗"
    ax.text(0.05, 0.95, f'd$\\rho$/d$P > 0$: {status}', transform=ax.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    print(f"{phase}-Fe: dρ/dP > 0? {status}")

plt.tight_layout()
plt.show()

---
## Edge Cases and Error Handling <a name="edge-cases"></a>

Test behavior at extreme conditions and error handling.

In [ ]:
print("Testing edge cases and error handling:")
print("=" * 80)

edge_cases = [
    ("Very low pressure", 1e4, 300),  # 0.01 MPa
    ("Very high pressure", 500*GPa, 3000),  # Beyond calibration range
    ("Very low temperature", 1*GPa, 1),  # Near absolute zero
    ("Very high temperature", 10*GPa, 10000),  # Beyond calibration
    ("Negative pressure", -1*GPa, 300),  # Unphysical
    ("Zero temperature", 1*GPa, 0),  # Absolute zero
]

for phase, eos in [('bcc', eos_bcc), ('fcc', eos_fcc)]:
    print(f"\n{phase}-Fe:")
    for description, P, T in edge_cases:
        try:
            rho = eos.density(P, T)
            print(f"  {description:25s}: ρ = {rho:.2f} kg/m³ (P={P/GPa:.1f} GPa, T={T:.0f} K)")
        except Exception as e:
            print(f"  {description:25s}: ERROR - {type(e).__name__}: {str(e)}")

---
## Comparison with Published Data <a name="comparison"></a>

Compare calculations with values reported in Dorogokupets et al. (2017).

In [ ]:
# Data points extracted from supplementary tables of the paper
# These are approximate values for demonstration

print("Comparison with published data from Dorogokupets et al. (2017):")
print("=" * 80)

# Example: bcc-Fe at 10 GPa and various temperatures
P_comp = 10 * GPa
published_data_bcc = [
    # (T [K], ρ [kg/m³], Cp [J/(kg·K)], α [K⁻¹])
    (300, 8270, 445, 2.8e-5),
    (1000, 8120, 590, 4.5e-5),
]

print(f"\nbcc-Fe at P = {P_comp/GPa:.0f} GPa:")
print(f"{'T (K)':>8s} {'Property':>20s} {'Calculated':>15s} {'Published':>15s} {'Diff (%)':>12s}")
print("-" * 80)

for T, rho_pub, cp_pub, alpha_pub in published_data_bcc:
    rho_calc = eos_bcc.density(P_comp, T)
    cp_calc = eos_bcc.isobaric_heat_capacity(P_comp, T)
    alpha_calc = eos_bcc.thermal_expansion(P_comp, T)
    
    for name, calc, pub in [('Density', rho_calc, rho_pub),
                             ('Cp', cp_calc, cp_pub),
                             ('α', alpha_calc, alpha_pub)]:
        diff_pct = 100 * (calc - pub) / pub
        status = "✓" if abs(diff_pct) < 5 else "⚠"
        print(f"{T:8.0f} {name:>20s} {calc:15.6e} {pub:15.6e} {diff_pct:11.2f}% {status}")

# Miozzi20 Iron EoS

This section provides comprehensive testing and benchmarking of the `Miozzi20` class implementation.

**Contents:**
1. [Setup and Imports](#setup-miozzi)
2. [Basic Functionality Tests](#basic-tests-miozzi)
3. [Reference Condition Validation](#reference-validation-miozzi)
4. [Convergence Tests](#convergence-miozzi)
5. [Runtime Benchmarking](#runtime-miozzi)
6. [Diagnostic Plots](#diagnostics-miozzi)
7. [Thermodynamic Consistency Tests](#consistency-miozzi)
8. [Edge Cases and Error Handling](#edge-cases-miozzi)
9. [Comparison with Published Data](#comparison-miozzi)

---

## Setup and Imports <a name="setup-miozzi"></a>

Import necessary libraries and initialize the Miozzi20 EoS instance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

from iron_eos import Miozzi20

# Set plotting style
plt.style.use('tableau-colorblind10')

# Physical constants
GPa = 1e9  # Conversion factor

print("✓ Imports successful")

In [ ]:
# Initialize Miozzi20 EoS instance for hcp-Fe
eos_hcp = Miozzi20()

print("✓ Miozzi20 EoS instance created")
print(f"  - hcp-Fe: V_0 = {eos_hcp.params['V0']*1e6:.4f} cm³/mol, K_0 = {eos_hcp.params['K0']/GPa:.1f} GPa")
print(f"  - K'_0 = {eos_hcp.params['K0_prime']:.2f}")
print(f"  - θ_D,0 = {eos_hcp.params['theta_D0']:.0f} K")
print(f"  - γ_0 = {eos_hcp.params['gamma0']:.3f}, q = {eos_hcp.params['q']:.2f}")
print(f"  - Reference temperature: T_0 = {eos_hcp.T0:.0f} K")

---
## Basic Functionality Tests <a name="basic-tests-miozzi"></a>

Test that all public methods execute without errors and return reasonable values.

We test at high-pressure conditions where hcp-Fe is stable (>60 GPa).

In [ ]:
# Test conditions: high pressure where hcp-Fe is stable
P_test = 100 * GPa  # 100 GPa
T_test = 2000  # K

print(f"Testing hcp-Fe at P = {P_test/GPa:.0f} GPa, T = {T_test:.0f} K:")
print("=" * 70)

methods = [
    ('density', 'kg/m³'),
    ('specific_internal_energy', 'J/kg'),
    ('specific_entropy', 'J/(kg·K)'),
    ('isobaric_heat_capacity', 'J/(kg·K)'),
    ('isochoric_heat_capacity', 'J/(kg·K)'),
    ('thermal_expansion', 'K⁻¹'),
    ('adiabatic_gradient', 'dimensionless')
]

results_hcp = {}
for method_name, units in methods:
    try:
        method = getattr(eos_hcp, method_name)
        result = method(P_test, T_test)
        results_hcp[method_name] = result
        print(f"  {method_name:30s}: {result:12.6e} {units}")
    except Exception as e:
        print(f"  {method_name:30s}: ERROR - {e}")
        results_hcp[method_name] = None

print("\n✓ All methods executed successfully" if all(v is not None for v in results_hcp.values()) 
      else "\n✗ Some methods failed")

In [ ]:
# Test at inner core boundary conditions
P_test_icb = 330 * GPa  # Inner core boundary
T_test_icb = 6000  # K (approximate ICB temperature)

print(f"\nTesting hcp-Fe at inner core boundary conditions:")
print(f"P = {P_test_icb/GPa:.0f} GPa, T = {T_test_icb:.0f} K")
print("=" * 70)

results_icb = {}
for method_name, units in methods:
    try:
        method = getattr(eos_hcp, method_name)
        result = method(P_test_icb, T_test_icb)
        results_icb[method_name] = result
        print(f"  {method_name:30s}: {result:12.6e} {units}")
    except Exception as e:
        print(f"  {method_name:30s}: ERROR - {e}")
        results_icb[method_name] = None

print("\n✓ All methods executed successfully" if all(v is not None for v in results_icb.values()) 
      else "\n✗ Some methods failed")

---
## Reference Condition Validation <a name="reference-validation-miozzi"></a>

Compare calculated properties with values from the Miozzi et al. (2020) paper.

In [ ]:
# Reference values for hcp-Fe from the paper
# At 127 GPa, 2600 K (from Figure 1b in the paper)
reference_hcp = {
    'pressure': 127 * GPa,  # Pa
    'temperature': 2600,  # K
    'volume': 16.625,  # Å³ (from paper)
    'density': 11860,  # kg/m³ (calculated from volume)
}

P_ref = reference_hcp['pressure']
T_ref = reference_hcp['temperature']

print(f"Validation against reference values from Miozzi et al. (2020):")
print(f"P = {P_ref/GPa:.0f} GPa, T = {T_ref:.0f} K")
print("=" * 80)

# Calculate density
rho_calc = eos_hcp.density(P_ref, T_ref)
rho_ref = reference_hcp['density']
diff_pct_rho = 100 * (rho_calc - rho_ref) / rho_ref

print(f"{'Property':30s} {'Calculated':>15s} {'Reference':>15s} {'Diff (%)':>12s}")
print("=" * 80)
print(f"{'Density':30s} {rho_calc:15.2f} {rho_ref:15.2f} {diff_pct_rho:11.2f}% \
{'✓' if abs(diff_pct_rho) < 5 else '⚠'}")

---
## Convergence Tests <a name="convergence-miozzi"></a>

Test the numerical convergence of the root-finding algorithm used to determine volume from pressure.

In [ ]:
# Test convergence by checking P(V(P,T), T) ≈ P
test_pressures = np.logspace(np.log10(60*GPa), np.log10(360*GPa), 25)  # 60-360 GPa range
test_temperatures = [300, 1500, 3000, 6000]  # K

print("Convergence test: P(V(P,T), T) - P should be ~ 0")
print("=" * 80)

fig, ax = plt.subplots(1, 1, figsize=(7, 4), dpi=300)

max_errors = []

for T in test_temperatures:
    errors = []
    for P in test_pressures:
        try:
            # Find volume for given P, T
            V = eos_hcp._find_volume(P, T)
            # Calculate pressure from that volume
            P_calc = eos_hcp._total_pressure(V, T)
            # Relative error
            rel_error = abs(P_calc - P) / P
            errors.append(rel_error)
        except:
            errors.append(np.nan)
    
    max_err = np.nanmax(errors)
    max_errors.append(max_err)
    
    ax.semilogy(test_pressures/GPa, errors, 'o-', label=f'$T = {T:.0f}$ K', linewidth=2, markersize=4)
    print(f"  T = {T:5.0f} K: max relative error = {max_err:.2e}")

ax.set_xlabel('Pressure [GPa]', fontsize=12)
ax.set_ylabel('Relative error in pressure', fontsize=12)
ax.grid(True, alpha=0.3)
ax.axhline(y=1e-5, color='r', linestyle='--', alpha=0.5, label='Target ($10^{-5}$)')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

if max(max_errors) < 1e-5:
    print("\n✓ Convergence test passed: all errors < target")
else:
    print(f"\n⚠ Some errors exceed the target: max = {max(max_errors):.2e}")

---
## Runtime Benchmarking <a name="runtime-miozzi"></a>

Measure execution time for various methods and conditions.

In [ ]:
# Runtime benchmarking
print("Runtime Benchmarking:")
print("=" * 80)

# Test parameters
P_bench = 150 * GPa
T_bench = 3000
n_calls = 1000

benchmark_results = {}

for method_name, _ in methods:
    method = getattr(eos_hcp, method_name)
    
    start_time = time.time()
    for _ in range(n_calls):
        result = method(P_bench, T_bench)
    end_time = time.time()
    
    avg_time = (end_time - start_time) / n_calls * 1000  # Convert to ms
    benchmark_results[method_name] = avg_time
    
    print(f"  {method_name:30s}: {avg_time:8.3f} ms per call")

print(f"\nTotal time for {n_calls} calls: {sum(benchmark_results.values()):.2f} ms")
print(f"Average time per method: {np.mean(list(benchmark_results.values())):.3f} ms")

In [ ]:
# Visualize timing results
fig, ax = plt.subplots(figsize=(7, 4), dpi=300)

methods_names = list(benchmark_results.keys())
times = list(benchmark_results.values())

x = np.arange(len(methods_names))
width = 0.35

ax.bar(x, times, width, alpha=0.8)

ax.set_ylabel('Time [ms/call]')
ax.set_xticks(x)
ax.set_xticklabels([m.replace('_', '\n') for m in methods_names], rotation=45, ha='right')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## Diagnostic Plots <a name="diagnostics-miozzi"></a>

Visualize EoS behavior across pressure-temperature space.

In [ ]:
# Pressure-Volume-Temperature surface
print("Generating P-V-T diagnostic plots...")

P_diag = np.linspace(60*GPa, 360*GPa, 30)
T_diag_lines = [300, 1500, 3000, 4500, 6000]

fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=300)

# 1. Density vs Pressure
ax = axes[0, 0]
for T in T_diag_lines:
    rho_vals = [eos_hcp.density(P, T) for P in P_diag]
    ax.plot(P_diag/GPa, rho_vals, linewidth=2, label=f'$T = {T:.0f}$ K')
ax.set_xlabel('Pressure [GPa]', fontsize=11)
ax.set_ylabel('Density [kg/m³]', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 2. Adiabatic gradient vs Pressure
ax = axes[0, 1]
for T in T_diag_lines:
    alpha_vals = [eos_hcp.adiabatic_gradient(P, T) for P in P_diag]
    ax.plot(P_diag/GPa, alpha_vals, linewidth=2, label=f'$T = {T:.0f}$ K')
ax.set_xlabel('Pressure [GPa]', fontsize=11)
ax.set_ylabel('Adiabatic gradient', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 3. Specific entropy vs Temperature
ax = axes[1, 0]
T_diag_range = np.linspace(300, 6000, 50)
P_diag_lines = [60*GPa, 120*GPa, 200*GPa, 330*GPa]
for P in P_diag_lines:
    S_vals = [eos_hcp.specific_entropy(P, T) for T in T_diag_range]
    ax.plot(T_diag_range, S_vals, linewidth=2, label=f'$P = {P/GPa:.0f}$ GPa')
ax.set_xlabel('Temperature [K]', fontsize=11)
ax.set_ylabel('Specific entropy [J/(kg·K)]', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 4. Heat capacity ratio Cp/Cv
ax = axes[1, 1]
for P in P_diag_lines:
    cp_cv_ratio = []
    for T in T_diag_range:
        cp = eos_hcp.isobaric_heat_capacity(P, T)
        cv = eos_hcp.isochoric_heat_capacity(P, T)
        cp_cv_ratio.append(cp/cv)
    ax.plot(T_diag_range, cp_cv_ratio, linewidth=2, label=f'$P = {P/GPa:.0f}$ GPa')
ax.set_xlabel('Temperature [K]', fontsize=11)
ax.set_ylabel('Heat capacity ratio Cp/Cv', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.axhline(y=1, color='r', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("✓ Diagnostic plots generated")

---
## Thermodynamic Consistency Tests <a name="consistency-miozzi"></a>

Verify that the EoS satisfies fundamental thermodynamic relations.

### Test: Cp ≥ Cv (thermodynamic stability)

In [ ]:
print("Testing Cp ≥ Cv (thermodynamic stability):")
print("=" * 80)

# Test at various conditions
test_conditions = [
    (60*GPa, 300),
    (100*GPa, 1500),
    (150*GPa, 3000),
    (200*GPa, 4500),
    (330*GPa, 6000),
]

violations = 0

for P, T in test_conditions:
    cp = eos_hcp.isobaric_heat_capacity(P, T)
    cv = eos_hcp.isochoric_heat_capacity(P, T)
    ratio = cp / cv
    
    status = "✓" if cp >= cv else "✗"
    if cp < cv:
        violations += 1
    
    print(f"  P={P/GPa:6.0f} GPa, T={T:5.0f} K: Cp/Cv = {ratio:.4f} {status}")

if violations == 0:
    print("\n✓ All tests passed: Cp ≥ Cv everywhere")
else:
    print(f"\n✗ Found {violations} violations of Cp ≥ Cv")

### Test: Entropy increases with temperature at constant pressure

In [ ]:
print("Testing dS/dT > 0 at constant pressure:")
print("=" * 80)

P_test_s = 150 * GPa
T_test_s = np.linspace(500, 6000, 50)

fig, ax = plt.subplots(figsize=(7, 4), dpi=300)

S_values = []
for T in T_test_s:
    S = eos_hcp.specific_entropy(P_test_s, T)
    S_values.append(S)

# Check monotonicity
dS = np.diff(S_values)
all_positive = np.all(dS > 0)

ax.plot(T_test_s, S_values, 'o-', linewidth=2, markersize=4)
ax.set_xlabel('Temperature [K]', fontsize=12)
ax.set_ylabel('Specific entropy [J/(kg·K)]', fontsize=12)
ax.grid(True, alpha=0.3)

status = "✓" if all_positive else "✗"
ax.text(0.05, 0.95, f'd$S$/d$T > 0$: {status}', transform=ax.transAxes,
        verticalalignment='top', fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

print(f"hcp-Fe: dS/dT > 0? {status}")

plt.tight_layout()
plt.show()

### Test: Density increases with pressure at constant temperature

In [ ]:
print("Testing dρ/dP > 0 at constant temperature:")
print("=" * 80)

T_test_rho = 3000  # K
P_test_rho = np.linspace(60*GPa, 360*GPa, 50)

fig, ax = plt.subplots(figsize=(7, 4), dpi=300)

rho_values = []
for P in P_test_rho:
    rho = eos_hcp.density(P, T_test_rho)
    rho_values.append(rho)

# Check monotonicity
drho = np.diff(rho_values)
all_positive = np.all(drho > 0)

ax.plot(P_test_rho/GPa, rho_values, 'o-', linewidth=2, markersize=4)
ax.set_xlabel('Pressure [GPa]', fontsize=12)
ax.set_ylabel('Density [kg/m³]', fontsize=12)
ax.grid(True, alpha=0.3)

status = "✓" if all_positive else "✗"
ax.text(0.05, 0.95, f'd$\\rho$/d$P > 0$: {status}', transform=ax.transAxes,
        verticalalignment='top', fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

print(f"hcp-Fe: dρ/dP > 0? {status}")

plt.tight_layout()
plt.show()

---
## Edge Cases and Error Handling <a name="edge-cases-miozzi"></a>

Test behavior at extreme conditions and error handling.

In [ ]:
print("Testing edge cases and error handling:")
print("=" * 80)

edge_cases = [
    ("Lower stability limit", 60*GPa, 300),  # Lower bound of hcp stability
    ("Very high pressure", 400*GPa, 6000),  # Beyond typical range
    ("Very low temperature", 100*GPa, 1),  # Near absolute zero
    ("Very high temperature", 150*GPa, 10000),  # Beyond calibration
    ("Zero temperature", 100*GPa, 0),  # Absolute zero
    ("ICB conditions", 330*GPa, 6000),  # Inner core boundary
]

print(f"\nhcp-Fe:")
for description, P, T in edge_cases:
    try:
        rho = eos_hcp.density(P, T)
        print(f"  {description:25s}: ρ = {rho:.2f} kg/m³ (P = {P/GPa:.0f} GPa, T = {T:.0f} K)")
    except Exception as e:
        print(f"  {description:25s}: ERROR - {type(e).__name__}: {str(e)}")

---
## Comparison with Published Data <a name="comparison-miozzi"></a>

Compare calculations with values reported in Miozzi et al. (2020).

We extract data points from Figure 4a (P-V-T data) of the paper.

In [ ]:
# Data points extracted from the paper (Figure 4a and supplementary data)
# These represent experimental P-V-T measurements

print("Comparison with published data from Miozzi et al. (2020):")
print("=" * 80)

# Example data points (P [GPa], T [K], V [Å³])
# Extracted from Figure 4a showing P-V-T isotherms
published_data_hcp = [
    # (P [GPa], T [K], expected density [kg/m³])
    (68, 1790, 10900),  # Approximate from lower P isotherms
    (100, 2300, 11600),  # Mid-range conditions
    (135, 2900, 12200),  # High P-T conditions
]

print(f"\nhcp-Fe comparison:")
print(f"{'P (GPa)':>8s} {'T (K)':>8s} {'Calculated':>15s} {'Published':>15s} {'Diff (%)':>12s}")
print("-" * 80)

for P_gpa, T, rho_pub in published_data_hcp:
    P = P_gpa * GPa
    rho_calc = eos_hcp.density(P, T)
    
    diff_pct = 100 * (rho_calc - rho_pub) / rho_pub
    status = "✓" if abs(diff_pct) < 10 else "⚠"
    print(f"{P_gpa:8.0f} {T:8.0f} {rho_calc:15.2f} {rho_pub:15.2f} {diff_pct:11.2f}% {status}")

In [ ]:
# Comparison with PREM density profile at core conditions
print("\n" + "=" * 80)
print("Comparison with PREM at inner core conditions:")
print("=" * 80)

# PREM density at ICB
P_icb = 330 * GPa
T_icb = 6000  # K (estimated ICB temperature)
rho_prem_icb = 12763  # kg/m³

rho_calc_icb = eos_hcp.density(P_icb, T_icb)
diff_prem = 100 * (rho_calc_icb - rho_prem_icb) / rho_prem_icb

print(f"\nInner Core Boundary (P = {P_icb/GPa:.0f} GPa, T = {T_icb:.0f} K):")
print(f"  Calculated density (pure Fe): {rho_calc_icb:.2f} kg/m³")
print(f"  PREM density:                 {rho_prem_icb:.2f} kg/m³")
print(f"  Difference:                   {diff_prem:+.2f}%")

# Hakim18 Iron EoS

This section provides comprehensive testing and benchmarking of the `Hakim18` class implementation.

**Contents:**
1. [Setup and Imports](#setup-hakim18)
2. [Basic Functionality Tests](#basic-tests-hakim18)
3. [Reference Condition Validation](#reference-validation-hakim18)
4. [Convergence Tests](#convergence-hakim18)
5. [Runtime Benchmarking](#runtime-hakim18)
6. [Diagnostic Plots](#diagnostics-hakim18)
7. [Thermodynamic Consistency Tests](#consistency-hakim18)
8. [Edge Cases and Error Handling](#edge-cases-hakim18)
9. [Comparison with Published Data](#comparison-hakim18)

---

## Setup and Imports <a name="setup-hakim18"></a>

Import necessary libraries and initialize the EoS instance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

from iron_eos import Hakim18

# Physical constants
GPa = 1e9   # Conversion factor for gigapascals
TPa = 1e12  # Conversion factor for terapascals

print("✓ Imports successful")

In [ ]:
# Initialize EoS instance for hcp-Fe
eos_hakim = Hakim18()

print("✓ Hakim18 EoS instance created")
print(f"  - Phase: hcp-Fe (super-Earth conditions)")
print(f"  - V_0 = {eos_hakim.params['V0']*1e6:.6f} cm³/mol")
print(f"  - P_0 = {eos_hakim.params['P0']/GPa:.1f} GPa (transition pressure)")
print(f"  - K_T,0 = {eos_hakim.params['KT0']/GPa:.1f} GPa")

---
## Basic Functionality Tests <a name="basic-tests-hakim18"></a>

Test that all public methods execute without errors and return reasonable values.

In [ ]:
# Test conditions: super-Earth relevant conditions
P_test = 0.5*TPa  # 0.5 TPa
T_test = 5000  # K

print(f"Testing hcp-Fe at super-Earth conditions (P = {P_test/GPa:.0f} GPa, T = {T_test:.0f} K):")
print("=" * 70)

methods = [
    ('density', 'kg/m³'),
    ('specific_internal_energy', 'J/kg'),
    ('specific_entropy', 'J/(kg·K)'),
    ('isobaric_heat_capacity', 'J/(kg·K)'),
    ('isochoric_heat_capacity', 'J/(kg·K)'),
    ('thermal_expansion', 'K⁻¹'),
    ('adiabatic_gradient', 'dimensionless')
]

results_hakim = {}
for method_name, units in methods:
    try:
        method = getattr(eos_hakim, method_name)
        result = method(P_test, T_test)
        results_hakim[method_name] = result
        print(f"  {method_name:30s}: {result:12.6e} {units}")
    except Exception as e:
        print(f"  {method_name:30s}: ERROR - {e}")
        results_hakim[method_name] = None

print("\n✓ All methods executed successfully" if all(v is not None for v in results_hakim.values()) 
      else "\n✗ Some methods failed")

In [ ]:
# Test at multiple pressure-temperature conditions across the valid range
test_conditions = [
    (234*GPa, 300, "Transition P, low T"),
    (500*GPa, 3000, "Moderate P, moderate T"),
    (1*TPa, 5000, "1 TPa, high T"),
    (5*TPa, 8000, "5 TPa, very high T"),
    (10*TPa, 10000, "10 TPa, extreme T"),
]

print("\nTesting across pressure-temperature range:")
print("=" * 90)
print(f"{'Condition':30s} {'P (GPa)':>10s} {'T (K)':>8s} {'ρ (kg/m³)':>12s} {'Status':>8s}")
print("=" * 90)

for P, T, description in test_conditions:
    try:
        rho = eos_hakim.density(P, T)
        status = "✓"
        print(f"{description:30s} {P/GPa:10.0f} {T:8.0f} {rho:12.2f} {status:>8s}")
    except Exception as e:
        status = "✗"
        print(f"{description:30s} {P/GPa:10.0f} {T:8.0f} {'ERROR':>12s} {status:>8s}")
        print(f"    Error: {e}")

---
## Reference Condition Validation <a name="reference-validation-hakim18"></a>

Compare calculated properties at the transition pressure with expected values.

In [ ]:
# Reference values at P_0 = 234.4 GPa (transition from experimental to DFT)
P_ref = 234.4 * GPa  # Pa
T_ref = 300  # K
V_ref = 4.28575e-6  # m³/mol (V_0 from the paper)

print("Validation at transition pressure (234.4 GPa, 300 K):")
print("=" * 80)

# Calculate density at reference conditions
rho_calc = eos_hakim.density(P_ref, T_ref)
rho_expected = 55.845e-3 / V_ref  # kg/m³

print(f"Density:")
print(f"  Calculated: {rho_calc:.2f} kg/m³")
print(f"  Expected:   {rho_expected:.2f} kg/m³")
print(f"  Difference: {abs(rho_calc - rho_expected)/rho_expected * 100:.4f}%")

if abs(rho_calc - rho_expected)/rho_expected < 0.01:
    print("  Status: ✓ (< 1% error)")
else:
    print("  Status: ⚠ (> 1% error)")

# Test pressure decomposition
V_test = eos_hakim._find_volume(P_ref, T_ref)
P_cold = eos_hakim._cold_pressure(V_test)
P_harm = eos_hakim._thermal_pressure_harmonic(V_test, T_ref)
P_ae = eos_hakim._thermal_pressure_anharmonic(V_test, T_ref)
P_total = eos_hakim._total_pressure(V_test, T_ref)

print(f"\nPressure decomposition at transition point:")
print(f"  Cold pressure:        {P_cold/GPa:8.2f} GPa ({P_cold/P_total*100:5.1f}%)")
print(f"  Harmonic thermal:     {P_harm/GPa:8.2f} GPa ({P_harm/P_total*100:5.1f}%)")
print(f"  Anharmonic-electronic:{P_ae/GPa:8.2f} GPa ({P_ae/P_total*100:5.1f}%)")
print(f"  Total:                {P_total/GPa:8.2f} GPa")
print(f"  Target:               {P_ref/GPa:8.2f} GPa")
print(f"  Error:                {abs(P_total - P_ref)/GPa:8.4f} GPa")

---
## Convergence Tests <a name="convergence-hakim18"></a>

Test the numerical convergence of the root-finding algorithm.

In [ ]:
# Test convergence by checking P(V(P,T), T) ≈ P
test_pressures = np.logspace(np.log10(234*GPa), np.log10(10*TPa), 30)
test_temperatures = [300, 2000, 5000, 8000]  # K

print("Convergence test: P(V(P,T), T) - P should be ~ 0")
print("=" * 80)

fig, ax = plt.subplots(figsize=(7, 4), dpi=300)

max_errors = []

for T in test_temperatures:
    errors = []
    valid_P = []
    
    for P in test_pressures:
        try:
            V = eos_hakim._find_volume(P, T)
            P_calc = eos_hakim._total_pressure(V, T)
            rel_error = abs(P_calc - P) / P
            errors.append(rel_error)
            valid_P.append(P)
        except Exception as e:
            print(f"  Warning: Failed at P={P/GPa:.0f} GPa, T={T}K: {e}")
            errors.append(np.nan)
    
    max_error = np.nanmax(errors) if len(errors) > 0 else np.nan
    max_errors.append(max_error)
    
    if len(valid_P) > 0:
        ax.loglog([p/GPa for p in valid_P], errors, 'o-', label=f'$T = {T}$ K', markersize=4)

ax.axhline(y=1e-5, color='k', linestyle='--', alpha=0.5, label='Target ($10^{-5}$)')
ax.set_xlabel('Pressure [GPa]', fontsize=12)
ax.set_ylabel('Relative error in pressure', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print(f"\nMaximum relative errors:")
for T, max_err in zip(test_temperatures, max_errors):
    status = "✓" if max_err < 1e-6 else "⚠"
    print(f"  T = {T:5.0f} K: {max_err:.3e} {status}")

if all(err < 1e-6 for err in max_errors if not np.isnan(err)):
    print("\n✓ All convergence tests passed (errors < target)")
else:
    print("\n⚠ Some convergence tests show larger errors")

---
## Runtime Benchmarking <a name="runtime-hakim18"></a>

Measure computational performance for various operations.

In [ ]:
print("Runtime benchmarking:")
print("=" * 80)

P_bench = 1 * TPa
T_bench = 5000  # K
n_iterations = 1000

print(f"Benchmark conditions: P = {P_bench/TPa:.0f} TPa, T = {T_bench:.0f} K")
print(f"Number of iterations: {n_iterations}\n")

benchmark_results = {}

for method_name, units in methods:
    method = getattr(eos_hakim, method_name)
    
    _ = method(P_bench, T_bench)
    
    start = time.perf_counter()
    for _ in range(n_iterations):
        _ = method(P_bench, T_bench)
    end = time.perf_counter()
    
    avg_time = (end - start) / n_iterations * 1e6
    benchmark_results[method_name] = avg_time
    
    print(f"  {method_name:30s}: {avg_time:8.2f} μs/call")

print(f"\n  Total for all properties:        {sum(benchmark_results.values()):8.2f} μs/call")

In [ ]:
# Visualize timing results
fig, ax = plt.subplots(figsize=(7, 4), dpi=300)

methods_names = list(benchmark_results.keys())
times = list(benchmark_results.values())

x = np.arange(len(methods_names))
width = 0.35

ax.bar(x, np.array(times)*1e-3, width, alpha=0.8)

ax.set_ylabel('Time [ms/call]')
ax.set_xticks(x)
ax.set_xticklabels([m.replace('_', '\n') for m in methods_names], rotation=45, ha='right')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## Diagnostic Plots <a name="diagnostics-hakim18"></a>

Visualize thermodynamic properties across the valid pressure-temperature range.

In [ ]:
# Pressure-Volume-Temperature surface
print("Generating P-V-T diagnostic plots...")

P_diag = np.linspace(300*GPa, 10*TPa, 50)
T_diag_lines = [300, 1500, 3000, 6000, 10000]

fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=300)

# 1. Density vs Pressure
ax = axes[0, 0]
for T in T_diag_lines:
    rho_vals = [eos_hakim.density(P, T) for P in P_diag]
    ax.plot(P_diag/TPa, rho_vals, linewidth=2, label=f'$T = {T:.0f}$ K')
ax.set_xlabel('Pressure [TPa]', fontsize=11)
ax.set_ylabel('Density [kg/m³]', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 2. Adiabatic gradient vs Pressure
ax = axes[0, 1]
for T in T_diag_lines:
    alpha_vals = [eos_hakim.adiabatic_gradient(P, T) for P in P_diag]
    ax.plot(P_diag/TPa, alpha_vals, linewidth=2, label=f'$T = {T:.0f}$ K')
ax.set_xlabel('Pressure [TPa]', fontsize=11)
ax.set_ylabel('Adiabatic gradient', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 3. Specific entropy vs Temperature
ax = axes[1, 0]
T_diag_range = np.linspace(300, 10000, 50)
P_diag_lines = [300*GPa, 1*TPa, 3*TPa, 10*TPa]
for P in P_diag_lines:
    S_vals = [eos_hakim.specific_entropy(P, T) for T in T_diag_range]
    ax.plot(T_diag_range, S_vals, linewidth=2, label=f'$P = {P/TPa:.1f}$ TPa')
ax.set_xlabel('Temperature [K]', fontsize=11)
ax.set_ylabel('Specific entropy [J/(kg·K)]', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 4. Heat capacity ratio Cp/Cv
ax = axes[1, 1]
for P in P_diag_lines:
    cp_cv_ratio = []
    for T in T_diag_range:
        cp = eos_hakim.isobaric_heat_capacity(P, T)
        cv = eos_hakim.isochoric_heat_capacity(P, T)
        cp_cv_ratio.append(cp/cv)
    ax.plot(T_diag_range, cp_cv_ratio, linewidth=2, label=f'$P = {P/TPa:.1f}$ TPa')
ax.set_xlabel('Temperature [K]', fontsize=11)
ax.set_ylabel('Heat capacity ratio Cp/Cv', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.axhline(y=1, color='r', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("✓ Diagnostic plots generated")

---
## Thermodynamic Consistency Tests <a name="consistency-hakim18"></a>

Verify that thermodynamic properties satisfy required relationships.

### Test: C_P > C_V

In [ ]:
print("Testing C_P > C_V:")
print("=" * 80)

P_test_c = np.linspace(234*GPa, 5*TPa, 50)
T_test_c = 5000  # K

fig, ax = plt.subplots(figsize=(7, 4), dpi=300)

Cp_values = []
Cv_values = []

for P in P_test_c:
    Cp = eos_hakim.isobaric_heat_capacity(P, T_test_c)
    Cv = eos_hakim.isochoric_heat_capacity(P, T_test_c)
    Cp_values.append(Cp)
    Cv_values.append(Cv)

all_positive = all(cp > cv for cp, cv in zip(Cp_values, Cv_values))

ax.plot(P_test_c/TPa, Cp_values, label='$C_P$', linewidth=2)
ax.plot(P_test_c/TPa, Cv_values, label='$C_V$', linewidth=2)

ax.set_xlabel('Pressure [TPa]', fontsize=12)
ax.set_ylabel('Heat capacity [J/(kg·K)]', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

status = "✓" if all_positive else "✗"
ax.text(0.05, 0.95, f'$C_P > C_V$: {status}', transform=ax.transAxes,
        verticalalignment='top', fontsize=12, 
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

print(f"hcp-Fe: C_P > C_V everywhere? {status}")

plt.tight_layout()
plt.show()

### Test: Entropy increases with temperature

In [ ]:
print("\nTesting dS/dT > 0 at constant pressure:")
print("=" * 80)

P_test_s = 1 * TPa
T_test_s = np.linspace(500, 10000, 50)

fig, ax = plt.subplots(figsize=(7, 4), dpi=300)

S_values = []
for T in T_test_s:
    S = eos_hakim.specific_entropy(P_test_s, T)
    S_values.append(S)

dS = np.diff(S_values)
all_positive = np.all(dS > 0)

ax.plot(T_test_s, S_values, linewidth=2)
ax.set_xlabel('Temperature [K]', fontsize=12)
ax.set_ylabel('Specific entropy [J/(kg·K)]', fontsize=12)
ax.grid(True, alpha=0.3)

status = "✓" if all_positive else "✗"
ax.text(0.05, 0.95, f'd$S$/d$T > 0$: {status}', transform=ax.transAxes,
        verticalalignment='top', fontsize=12, 
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

print(f"hcp-Fe: dS/dT > 0? {status}")

plt.tight_layout()
plt.show()

### Test: Density increases with pressure

In [ ]:
print("\nTesting dρ/dP > 0 at constant temperature:")
print("=" * 80)

T_test_rho = 5000  # K
P_test_rho = np.linspace(234*GPa, 10*TPa, 50)

fig, ax = plt.subplots(figsize=(7, 4), dpi=300)

rho_values = []
for P in P_test_rho:
    rho = eos_hakim.density(P, T_test_rho)
    rho_values.append(rho)

drho = np.diff(rho_values)
all_positive = np.all(drho > 0)

ax.plot(P_test_rho/TPa, rho_values, linewidth=2)
ax.set_xlabel('Pressure [TPa]', fontsize=12)
ax.set_ylabel('Density [kg/m³]', fontsize=12)
ax.grid(True, alpha=0.3)

status = "✓" if all_positive else "✗"
ax.text(0.05, 0.95, f'd$\\rho$/d$P > 0$: {status}', transform=ax.transAxes,
        verticalalignment='top', fontsize=12, 
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

print(f"hcp-Fe: dρ/dP > 0? {status}")

plt.tight_layout()
plt.show()

---
## Edge Cases and Error Handling <a name="edge-cases-hakim18"></a>

Test behavior at extreme conditions and error handling.

In [ ]:
print("Testing edge cases and error handling:")
print("=" * 80)

edge_cases = [
    ("Transition pressure", 234*GPa, 300),
    ("Low temperature", 500*GPa, 1),
    ("Very high temperature", 1*TPa, 15000),
    ("Moderate super-Earth", 1*TPa, 5000),
    ("Large super-Earth", 5*TPa, 8000),
    ("Ultra-massive planet", 10*TPa, 10000),
    ("Beyond DFT range", 20*TPa, 10000),
    ("Zero temperature", 1*TPa, 1e-10),
]

print(f"\nhcp-Fe (Hakim18):")
for description, P, T in edge_cases:
    try:
        rho = eos_hakim.density(P, T)
        print(f"  {description:25s}: ρ = {rho:.2f} kg/m³ (P = {P/TPa:.1f} TPa, T = {T:.0f} K)")
    except Exception as e:
        print(f"  {description:25s}: ERROR - {type(e).__name__}: {str(e)[:60]}")

---
## Comparison with Published Data <a name="comparison-hakim18"></a>

Compare calculations with values from Hakim et al. (2018).

In [ ]:
print("Comparison with DFT data from Hakim et al. (2018):")
print("=" * 80)

published_data = [
    (0.234, 300, 13000),
    (1.0, 300, 18500),
    (1.0, 5000, 17800),
]

print(f"\nComparison with paper data:")
print(f"{'P (TPa)':>8s} {'T (K)':>8s} {'Calculated':>15s} {'Published':>15s} {'Diff (%)':>12s}")
print("-" * 80)

for P_tpa, T, rho_pub in published_data:
    P = P_tpa * TPa
    try:
        rho_calc = eos_hakim.density(P, T)
        
        diff_pct = 100 * (rho_calc - rho_pub) / rho_pub
        status = "✓" if abs(diff_pct) < 10 else "⚠"
        print(f"{P_tpa:8.1f} {T:8.0f} {rho_calc:15.2f} {rho_pub:15.2f} {diff_pct:11.2f}% {status}")
    except Exception as e:
        print(f"{P_tpa:8.1f} {T:8.0f} {'ERROR':>15s} {rho_pub:15.2f} {'N/A':>12s}")

# Luo24 Iron EoS

This section provides comprehensive testing and benchmarking of the `Luo24` class implementation for liquid iron.

**Contents:**
1. [Setup and Imports](#setup-luo24)
2. [Basic Functionality Tests](#basic-tests-luo24)
3. [Reference Condition Validation](#reference-validation-luo24)
4. [Convergence Tests](#convergence-luo24)
5. [Runtime Benchmarking](#runtime-luo24)
6. [Diagnostic Plots](#diagnostics-luo24)
7. [Thermodynamic Consistency Tests](#consistency-luo24)
8. [Edge Cases and Error Handling](#edge-cases-luo24)
9. [Comparison with Published Data](#comparison-luo24)

---

## Setup and Imports <a name="setup-luo24"></a>

Import necessary libraries and initialize the Luo24 EoS instance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

from iron_eos import Luo24, Ichikawa20

# Set plotting style
plt.style.use('tableau-colorblind10')

# Physical constants
GPa = 1e9   # Conversion factor for gigapascals
TPa = 1e12  # Conversion factor for terapascals

print("✓ Imports successful")

In [ ]:
# Initialize Luo24 EoS instance for liquid iron
eos_luo = Luo24()
#eos_luo = Ichikawa20()

print("✓ Luo24 EoS instance created")
print(f"  - Phase: liquid Fe (super-Earth conditions)")
print(f"  - V_0 = {eos_luo.params['V0']*1e6:.6f} cm³/mol")
print(f"  - T_0 = {eos_luo.T0:.0f} K (reference temperature)")
print(f"  - K_0 = {eos_luo.params['K0']/GPa:.2f} GPa")
print(f"  - K'_0 = {eos_luo.params['K0_prime']:.3f}")
print(f"  - Thermal coefficients: a = {eos_luo.params['a']/GPa:.3f}, \
b = {eos_luo.params['b']/GPa:.3f}, c = {eos_luo.params['c']/GPa:.3f} GPa")

---
## Basic Functionality Tests <a name="basic-tests-luo24"></a>

Test that all public methods execute without errors and return reasonable values.

The Luo24 EoS is calibrated for liquid iron at super-Earth core conditions: ~50-1300 GPa and ~8000-14000 K.

In [ ]:
# Test conditions: super-Earth liquid core conditions (near reference T_0)
P_test = 500 * GPa  # 500 GPa
T_test = 9000  # K

print(f"Testing liquid Fe at super-Earth conditions (P = {P_test/GPa:.0f} GPa, T = {T_test:.0f} K):")
print("=" * 70)

methods = [
    ('density', 'kg/m³'),
    ('specific_internal_energy', 'J/kg'),
    ('specific_entropy', 'J/(kg·K)'),
    ('isobaric_heat_capacity', 'J/(kg·K)'),
    ('isochoric_heat_capacity', 'J/(kg·K)'),
    ('thermal_expansion', 'K⁻¹'),
    ('adiabatic_gradient', 'dimensionless')
]

results_luo = {}
for method_name, units in methods:
    try:
        method = getattr(eos_luo, method_name)
        result = method(P_test, T_test)
        results_luo[method_name] = result
        print(f"  {method_name:30s}: {result:12.6e} {units}")
    except Exception as e:
        print(f"  {method_name:30s}: ERROR - {e}")
        results_luo[method_name] = None

print("\n✓ All methods executed successfully" if all(v is not None for v in results_luo.values()) 
      else "\n✗ Some methods failed")

In [ ]:
# Test at multiple pressure-temperature conditions across the valid range
test_conditions = [
    (100*GPa, 8000, "Low P, reference T"),
    (250*GPa, 6500, "250 GPa, 6500 K (from paper)"),
    (500*GPa, 9000, "500 GPa, 9000 K (from paper)"),
    (1*TPa, 13000, "1 TPa, 13000 K (from paper)"),
    (1.3*TPa, 14000, "1.3 TPa, 14000 K (extreme)"),
]

print("\nTesting across pressure-temperature range:")
print("=" * 90)
print(f"{'Condition':30s} {'P (GPa)':>10s} {'T (K)':>8s} {'ρ (kg/m³)':>12s} {'Status':>8s}")
print("=" * 90)

for P, T, description in test_conditions:
    try:
        rho = eos_luo.density(P, T)
        status = "✓"
        print(f"{description:30s} {P/GPa:10.0f} {T:8.0f} {rho:12.2f} {status:>8s}")
    except Exception as e:
        status = "✗"
        print(f"{description:30s} {P/GPa:10.0f} {T:8.0f} {'ERROR':>12s} {status:>8s}")
        print(f"    Error: {e}")

---
## Reference Condition Validation <a name="reference-validation-luo24"></a>

Compare calculated properties at reference conditions (T_0 = 8000 K) with expected values from the paper.

From the Luo et al. (2024) paper, the EoS parameters are:
- V_0 = 1043.912 Å³ (for 64 Fe atoms, at T_0 = 8000 K)
- K_{T0} = 49.249 GPa
- K'_{T0} = 4.976

In [ ]:
# At reference temperature T_0 = 8000 K, the thermal pressure should be zero
# So P(V_0, T_0) should equal P_cold(V_0) = 0 (at reference volume)

T_ref = eos_luo.T0  # 8000 K
V0 = eos_luo.params['V0']

print("Validation at reference temperature (T_0 = 8000 K):")
print("=" * 80)

# Check pressure at reference volume and temperature
P_cold_at_V0 = eos_luo._cold_pressure(V0)
P_th_at_V0_T0 = eos_luo._thermal_pressure(V0, T_ref)
P_total_at_V0_T0 = eos_luo._total_pressure(V0, T_ref)

print(f"\nPressure at V_0, T_0:")
print(f"  V_0 = {V0*1e6:.6f} cm³/mol")
print(f"  T_0 = {T_ref:.0f} K")
print(f"  P_cold(V_0) = {P_cold_at_V0/GPa:.4f} GPa")
print(f"  P_th(V_0, T_0) = {P_th_at_V0_T0/GPa:.4f} GPa (should be ~0)")
print(f"  P_total = {P_total_at_V0_T0/GPa:.4f} GPa")

# Check density at reference conditions
rho_at_ref = 55.845e-3 / V0  # kg/m³ (expected from V_0)
print(f"\nExpected density at V_0: {rho_at_ref:.2f} kg/m³")

In [ ]:
# Verify bulk modulus at reference conditions
# K_T at V_0, T_0 should be close to K_0 = 49.249 GPa

K_T_calc = eos_luo._isothermal_bulk_modulus(V0, T_ref)
K_0_expected = eos_luo.params['K0']

print(f"\nBulk modulus at reference conditions:")
print(f"  K_T(V_0, T_0) calculated: {K_T_calc/GPa:.3f} GPa")
print(f"  K_0 expected:             {K_0_expected/GPa:.3f} GPa")
print(f"  Difference:               {abs(K_T_calc - K_0_expected)/K_0_expected * 100:.2f}%")

if abs(K_T_calc - K_0_expected)/K_0_expected < 0.01:
    print("  Status: ✓ (< 1% error)")
else:
    print("  Status: ⚠ (> 1% error)")

---
## Convergence Tests <a name="convergence-luo24"></a>

Test the numerical convergence of the root-finding algorithm used to determine volume from pressure.

In [ ]:
# Test convergence by checking P(V(P,T), T) ≈ P
test_pressures = np.logspace(np.log10(50*GPa), np.log10(1.3*TPa), 30)
test_temperatures = [6500, 8000, 9000, 13000]  # K (from paper conditions)

print("Convergence test: P(V(P,T), T) ≈ P")
print("=" * 80)

max_errors = {}
for T in test_temperatures:
    errors = []
    for P in test_pressures:
        try:
            V = eos_luo._find_volume(P, T)
            P_calc = eos_luo._total_pressure(V, T)
            rel_error = abs(P_calc - P) / P
            errors.append(rel_error)
        except:
            errors.append(np.nan)
    
    max_error = np.nanmax(errors)
    max_errors[T] = max_error
    status = "✓" if max_error < 1e-5 else "⚠"
    print(f"  T = {T:5.0f} K: max relative error = {max_error:.2e} {status}")

print(f"\nOverall max error: {max(max_errors.values()):.2e}")

In [ ]:
# Visualize convergence across P-T space
fig, ax = plt.subplots(figsize=(8, 5), dpi=300)

for T in test_temperatures:
    errors = []
    valid_P = []
    for P in test_pressures:
        try:
            V = eos_luo._find_volume(P, T)
            P_calc = eos_luo._total_pressure(V, T)
            rel_error = abs(P_calc - P) / P
            errors.append(rel_error)
            valid_P.append(P)
        except:
            pass
    
    ax.semilogy(np.array(valid_P)/GPa, errors, 'o-', markersize=3, label=f'$T = {T:.0f}$ K')

ax.axhline(y=1e-5, color='r', linestyle='--', label='Target')
ax.set_xlabel('Pressure [GPa]')
ax.set_ylabel('Relative error in pressure')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Runtime Benchmarking <a name="runtime-luo24"></a>

Measure computational performance for various operations.

In [ ]:
print("Runtime benchmarking:")
print("=" * 80)

P_bench = 500 * GPa
T_bench = 9000  # K
n_iterations = 1000

print(f"Benchmark conditions: P = {P_bench/GPa:.0f} GPa, T = {T_bench:.0f} K")
print(f"Number of iterations: {n_iterations}")
print()

benchmark_results = {}

for method_name, _ in methods:
    method = getattr(eos_luo, method_name)
    
    # Warmup
    for _ in range(10):
        method(P_bench, T_bench)
    
    # Timing
    start = time.perf_counter()
    for _ in range(n_iterations):
        method(P_bench, T_bench)
    elapsed = time.perf_counter() - start
    
    time_per_call = elapsed / n_iterations * 1e6  # microseconds
    benchmark_results[method_name] = time_per_call
    print(f"  {method_name:30s}: {time_per_call:8.2f} µs/call")

print(f"\nTotal time for {n_iterations} calls of all methods: \
{sum(benchmark_results.values())*n_iterations/1e6:.3f} s")

In [ ]:
# Visualize timing results
fig, ax = plt.subplots(figsize=(8, 5), dpi=300)

methods_names = list(benchmark_results.keys())
times = list(benchmark_results.values())

x = np.arange(len(methods_names))
width = 0.6

bars = ax.bar(x, np.array(times), width, alpha=0.8, color='steelblue')

ax.set_ylabel('Time [µs/call]')
ax.set_xticks(x)
ax.set_xticklabels([m.replace('_', '\n') for m in methods_names], rotation=45, ha='right', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## Diagnostic Plots <a name="diagnostics-luo24"></a>

Visualize thermodynamic properties across the valid pressure-temperature range (50 GPa - 1.3 TPa, 6000-14000 K).

In [ ]:
# Pressure-Volume-Temperature relationships
print("Generating P-V-T diagnostic plots...")

P_diag = np.linspace(50*GPa, 1.2*TPa, 50)
T_diag_lines = [6500, 8000, 9000, 11000, 13000]

fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=300)

# Density vs Pressure
ax = axes[0, 0]
for T in T_diag_lines:
    rho_vals = [eos_luo.density(P, T) for P in P_diag]
    ax.plot(P_diag/GPa, rho_vals, label=f'$T = {T:.0f}$ K')
ax.set_xlabel('Pressure [GPa]')
ax.set_ylabel('Density [kg/m³]')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Density vs Temperature (isobars)
ax = axes[0, 1]
T_range = np.linspace(6000, 14000, 50)
P_isobars = [100*GPa, 300*GPa, 500*GPa, 800*GPa, 1*TPa]
for P in P_isobars:
    rho_vals = [eos_luo.density(P, T) for T in T_range]
    ax.plot(T_range, rho_vals, label=f'$P = {P/GPa:.0f}$ GPa')
ax.set_xlabel('Temperature [K]')
ax.set_ylabel('Density [kg/m³]')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Internal energy vs Temperature
ax = axes[1, 0]
for P in P_isobars:
    E_vals = [eos_luo.specific_internal_energy(P, T)/1e6 for T in T_range]
    ax.plot(T_range, E_vals, label=f'$P = {P/GPa:.0f}$ GPa')
ax.set_xlabel('Temperature [K]')
ax.set_ylabel('Specific internal energy [MJ/kg]')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Entropy vs Temperature
ax = axes[1, 1]
for P in P_isobars:
    S_vals = [eos_luo.specific_entropy(P, T) for T in T_range]
    ax.plot(T_range, S_vals, label=f'$P = {P/GPa:.0f}$ GPa')
ax.set_xlabel('Temperature [K]')
ax.set_ylabel('Specific entropy [J/(kg·K)]')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Heat capacities and derived quantities
fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=300)

# C_P and C_V vs Pressure
ax = axes[0, 0]
T_cp = 9000  # K
Cp_vals = [eos_luo.isobaric_heat_capacity(P, T_cp) for P in P_diag]
Cv_vals = [eos_luo.isochoric_heat_capacity(P, T_cp) for P in P_diag]
ax.plot(P_diag/GPa, Cp_vals, 'b-', label='$C_P$', linewidth=2)
ax.plot(P_diag/GPa, Cv_vals, 'r--', label='$C_V$', linewidth=2)
ax.set_xlabel('Pressure [GPa]')
ax.set_ylabel('Heat capacity [J/(kg·K)]')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Thermal expansion vs Pressure
ax = axes[0, 1]
for T in T_diag_lines:
    alpha_vals = [eos_luo.thermal_expansion(P, T) for P in P_diag]
    ax.semilogy(P_diag/GPa, alpha_vals, label=f'$T = {T:.0f}$ K')
ax.set_xlabel('Pressure [GPa]')
ax.set_ylabel('Thermal expansion [K⁻¹]')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Adiabatic gradient vs Pressure
ax = axes[1, 0]
for T in T_diag_lines:
    grad_vals = [eos_luo.adiabatic_gradient(P, T) for P in P_diag]
    ax.plot(P_diag/GPa, grad_vals, label=f'$T = {T:.0f}$ K')
ax.set_xlabel('Pressure [GPa]')
ax.set_ylabel('Adiabatic gradient')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Gamma ratio (C_P/C_V) vs Pressure
ax = axes[1, 1]
for T in T_diag_lines:
    gamma_vals = [eos_luo.isobaric_heat_capacity(P, T)/eos_luo.isochoric_heat_capacity(P, T) for P in P_diag]
    ax.plot(P_diag/GPa, gamma_vals, label=f'$T = {T:.0f}$ K')
ax.set_xlabel('Pressure [GPa]')
ax.set_ylabel('$\\gamma = C_P/C_V$')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# P-T phase space heatmap for density
n_P = 40
n_T = 40
P_grid = np.linspace(50*GPa, 1.2*TPa, n_P)
T_grid = np.linspace(6000, 14000, n_T)
P_mesh, T_mesh = np.meshgrid(P_grid, T_grid)

rho_grid = np.zeros((n_T, n_P))
for i, T in enumerate(T_grid):
    for j, P in enumerate(P_grid):
        try:
            rho_grid[i, j] = eos_luo.density(P, T)
        except:
            rho_grid[i, j] = np.nan

fig, ax = plt.subplots(figsize=(10, 7), dpi=300)

c = ax.contourf(P_mesh/GPa, T_mesh, rho_grid, levels=30, cmap='viridis')
cbar = plt.colorbar(c, ax=ax)
cbar.set_label('Density [kg/m³]', fontsize=11)

# Mark the simulation conditions from the paper
paper_conditions = [(250, 6500), (500, 9000), (1000, 13000)]
for P_paper, T_paper in paper_conditions:
    ax.plot(P_paper, T_paper, 'w*', markersize=12, markeredgecolor='k')

ax.set_xlabel('Pressure [GPa]', fontsize=12)
ax.set_ylabel('Temperature [K]', fontsize=12)

plt.tight_layout()
plt.show()

---
## Thermodynamic Consistency Tests <a name="consistency-luo24"></a>

Verify that thermodynamic properties satisfy required relationships.

### Test: C_P ≥ C_V (thermodynamic stability)

In [ ]:
print("Testing C_P ≥ C_V (thermodynamic stability):")
print("=" * 80)

P_test_c = np.linspace(50*GPa, 1.2*TPa, 50)
T_test_c = 9000  # K

fig, ax = plt.subplots(figsize=(8, 5), dpi=300)

Cp_values = []
Cv_values = []
violations = 0

for P in P_test_c:
    try:
        Cp = eos_luo.isobaric_heat_capacity(P, T_test_c)
        Cv = eos_luo.isochoric_heat_capacity(P, T_test_c)
        Cp_values.append(Cp)
        Cv_values.append(Cv)
        if Cp < Cv:
            violations += 1
    except:
        Cp_values.append(np.nan)
        Cv_values.append(np.nan)

ax.plot(P_test_c/GPa, Cp_values, 'b-', linewidth=2, label='$C_P$')
ax.plot(P_test_c/GPa, Cv_values, 'r--', linewidth=2, label='$C_V$')
ax.set_xlabel('Pressure [GPa]')
ax.set_ylabel('Heat capacity [J/(kg·K)]')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

status = "✓" if violations == 0 else "✗"
ax.text(0.05, 0.95, f'C$_P$ ≥ C$_V$: {status} ({violations} violations)', 
        transform=ax.transAxes, verticalalignment='top', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

print(f"  Violations: {violations} / {len(P_test_c)} {status}")

plt.tight_layout()
plt.show()

### Test: Entropy increases with temperature at constant pressure

In [ ]:
print("\nTesting dS/dT > 0 at constant pressure:")
print("=" * 80)

P_test_s = 500 * GPa
T_test_s = np.linspace(6000, 14000, 50)

fig, ax = plt.subplots(figsize=(8, 5), dpi=300)

S_values = []
for T in T_test_s:
    try:
        S = eos_luo.specific_entropy(P_test_s, T)
        S_values.append(S)
    except:
        S_values.append(np.nan)

ax.plot(T_test_s, S_values, linewidth=2)
ax.set_xlabel('Temperature [K]')
ax.set_ylabel('Specific entropy [J/(kg·K)]')
ax.grid(True, alpha=0.3)

# Check monotonicity
dS = np.diff(S_values)
all_positive = np.all(dS > 0)
status = "✓" if all_positive else "✗"
ax.text(0.05, 0.95, f'd$S$/d$T$ > 0: {status}', transform=ax.transAxes,
        verticalalignment='top', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

print(f"  dS/dT > 0: {status}")

plt.tight_layout()
plt.show()

### Test: Density increases with pressure at constant temperature

In [ ]:
print("\nTesting dρ/dP > 0 at constant temperature:")
print("=" * 80)

T_test_rho = 9000  # K
P_test_rho = np.linspace(50*GPa, 1.2*TPa, 50)

fig, ax = plt.subplots(figsize=(8, 5), dpi=300)

rho_values = []
for P in P_test_rho:
    try:
        rho = eos_luo.density(P, T_test_rho)
        rho_values.append(rho)
    except:
        rho_values.append(np.nan)

ax.plot(P_test_rho/GPa, rho_values, linewidth=2)
ax.set_xlabel('Pressure [GPa]')
ax.set_ylabel('Density [kg/m³]')
ax.grid(True, alpha=0.3)

# Check monotonicity
drho = np.diff(rho_values)
all_positive = np.all(drho > 0)
status = "✓" if all_positive else "✗"
ax.text(0.05, 0.95, f'd$ρ$/d$P$ > 0: {status}', transform=ax.transAxes,
        verticalalignment='top', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

print(f"  dρ/dP > 0: {status}")

plt.tight_layout()
plt.show()

---
## Edge Cases and Error Handling <a name="edge-cases-luo24"></a>

Test behavior at extreme conditions and error handling.

The Luo24 EoS is calibrated for ~50-1300 GPa and ~8000-14000 K based on ab initio molecular dynamics simulations.

In [ ]:
print("Testing edge cases and error handling:")
print("=" * 80)

edge_cases = [
    ("Calibration lower bound", 50*GPa, 8000),
    ("Calibration upper bound", 1.3*TPa, 14000),
    ("250 GPa, 6500 K (paper)", 250*GPa, 6500),
    ("500 GPa, 9000 K (paper)", 500*GPa, 9000),
    ("1000 GPa, 13000 K (paper)", 1*TPa, 13000),
    ("Low temperature (near solidus)", 200*GPa, 5000),
    ("Very high temperature", 500*GPa, 20000),
    ("Beyond P range (high)", 2*TPa, 14000),
    ("Below P range (low)", 20*GPa, 8000),
]

print(f"\nLiquid Fe (Luo24):")
for description, P, T in edge_cases:
    try:
        rho = eos_luo.density(P, T)
        print(f"  {description:35s}: ρ = {rho:.2f} kg/m³ (P = {P/GPa:.0f} GPa, T = {T:.0f} K)")
    except Exception as e:
        print(f"  {description:35s}: ERROR - {type(e).__name__}: {str(e)[:50]}")

---
## Comparison with Published Data <a name="comparison-luo24"></a>

Compare calculations with values from Luo et al. (2024).

The paper provides simulation conditions and expected densities in Extended Data Table 1 and the Methods section.

In [ ]:
print("Comparison with ab initio MD data from Luo et al. (2024):")
print("=" * 80)

# Data from Extended Data Table 1 and Methods section
# V is given in Å³/atom, convert to molar volume for density calculation
# ρ = M / V where V is converted from Å³/atom to m³/mol

# From Extended Data Table 1:
# (P [GPa], T [K], V [Å³/atom])
published_data = [
    (250, 6500, 7.544),   # Fe64 at 250 GPa, 6500 K
    (500, 9000, 6.434),   # Fe64 at 500 GPa, 9000 K  
    (1000, 13000, 5.353), # Fe64 at 1000 GPa, 13000 K
]

# Atomic mass of Fe
M_Fe = 55.845e-3  # kg/mol
N_A = 6.02214076e23  # Avogadro

print(f"\nComparison with simulation data:")
print(f"{'P (GPa)':>8s} {'T (K)':>8s} {'Calc ρ':>12s} {'Pub ρ':>12s} {'Diff (%)':>10s}")
print("-" * 60)

for P_gpa, T, V_atom in published_data:
    P = P_gpa * GPa
    
    # Convert V from Å³/atom to m³/mol
    V_molar_pub = V_atom * 1e-30 * N_A  # m³/mol
    rho_pub = M_Fe / V_molar_pub  # kg/m³
    
    try:
        rho_calc = eos_luo.density(P, T)
        diff_pct = 100 * (rho_calc - rho_pub) / rho_pub
        status = "✓" if abs(diff_pct) < 5 else "⚠"
        print(f"{P_gpa:8.0f} {T:8.0f} {rho_calc:12.2f} {rho_pub:12.2f} {diff_pct:9.2f}% {status}")
    except Exception as e:
        print(f"{P_gpa:8.0f} {T:8.0f} {'ERROR':>12s} {rho_pub:12.2f} {'N/A':>10s}")

In [ ]:
# Visualize comparison with published data
fig, ax = plt.subplots(figsize=(8, 6), dpi=300)

# Calculate EoS predictions across P range for paper temperatures
P_range = np.linspace(100*GPa, 1.2*TPa, 50)

for T in [6500, 9000, 13000]:
    rho_eos = [eos_luo.density(P, T) for P in P_range]
    ax.plot(P_range/GPa, rho_eos, '-', label=f'EoS $T = {T}$ K')

# Plot published data points
M_Fe = 55.845e-3
N_A = 6.02214076e23

for P_gpa, T, V_atom in published_data:
    V_molar = V_atom * 1e-30 * N_A
    rho_pub = M_Fe / V_molar
    color = {'6500': 'C0', '9000': 'C1', '13000': 'C2'}[str(T)]
    ax.plot(P_gpa, rho_pub, 'o', markersize=10, color=color, 
            markeredgecolor='black', markeredgewidth=1.5,
            label=f'MD data $T = {T}$ K')

ax.set_xlabel('Pressure [GPa]', fontsize=12)
ax.set_ylabel('Density [kg/m³]', fontsize=12)
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Iron Phase Diagram

This section provides comprehensive testing and benchmarking of the phase diagram functions.

**Contents:**
1. [Setup and Imports](#setup-phase)
2. [Phase Boundary Functions](#phase-boundaries)
3. [Phase Diagram Visualization](#phase-diagram-viz)
4. [Triple Point Verification](#triple-point)
5. [Phase Determination Tests](#phase-determination)
6. [EoS Selection Integration](#eos-selection)
7. [Runtime Benchmarking](#runtime-phase)
8. [Reference Point Determination](#ref-point)
9. [Thermodynamic Properties Maps](#thermo-maps)

---

## Setup and Imports <a name="setup-phase"></a>

Import phase diagram functions and EoS classes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

from iron_eos import (
    T_melt_Fe, T_gamma_epsilon, T_alpha_gamma, T_delta_gamma, T_alpha_epsilon,
    get_iron_phase, get_iron_eos, get_iron_eos_for_PT,
    Dorogokupets17, Miozzi20, Hakim18, Luo24, Ichikawa20, HcpIronEos
)

# Set plotting style
plt.style.use('tableau-colorblind10')

# Physical constants
GPa = 1e9

print("✓ Imports successful")

---
## Phase Boundary Functions <a name="phase-boundaries"></a>

Test the individual phase boundary functions from Dorogokupets et al. (2017) and Anzellini et al. (2013).

In [ ]:
print("Testing phase boundary functions:")
print("=" * 80)

# Test at specific pressures
test_pressures_low = [0, 5, 7.3, 10, 15.8]  # GPa - low pressure regime
test_pressures_high = [50, 98.5, 150, 200, 300]  # GPa - high pressure regime

print("\nLow-pressure phase boundaries:")
print(f"{'P (GPa)':>8s} {'T_αγ (K)':>12s} {'T_δγ (K)':>12s} {'T_γε (K)':>12s} {'T_αε (K)':>12s} \
{'T_melt (K)':>12s}")
print("-" * 72)

for P_gpa in test_pressures_low:
    P = P_gpa * GPa
    T_ag = T_alpha_gamma(P) if P_gpa <= 7.3 else float('nan')
    T_dg = T_delta_gamma(P) if P_gpa <= 5.2 else float('nan')
    T_ge = T_gamma_epsilon(P)
    T_ae = T_alpha_epsilon(P) if 7.3 <= P_gpa <= 15.8 else float('nan')
    T_m = T_melt_Fe(P)
    
    print(f"{P_gpa:8.1f} {T_ag:12.1f} {T_dg:12.1f} {T_ge:12.1f} {T_ae:12.1f} {T_m:12.1f}")

print("\nHigh-pressure regime (only γ-ε and melting relevant):")
print(f"{'P (GPa)':>8s} {'T_γε (K)':>12s} {'T_melt (K)':>12s}")
print("-" * 36)

for P_gpa in test_pressures_high:
    P = P_gpa * GPa
    T_ge = T_gamma_epsilon(P)
    T_m = T_melt_Fe(P)
    print(f"{P_gpa:8.1f} {T_ge:12.1f} {T_m:12.1f}")

In [ ]:
# Verify known reference values from BICEPS paper
print("\nVerification against BICEPS paper reference values:")
print("=" * 80)

reference_values = [
    ('T_αγ at P = 0', T_alpha_gamma(0), 1120, 'Eq. 8: T_αγ(0) = 1120 K'),
    ('T_αγ at P = 7.3 GPa', T_alpha_gamma(7.3*GPa), 820, 'Eq. 8: T_αγ(7.3) = 820 K'),
    ('T_δγ at P = 0', T_delta_gamma(0), 1580, 'Eq. 9: T_δγ(0) = 1580 K'),
    ('T_δγ at P = 5.2 GPa', T_delta_gamma(5.2*GPa), 1998, 'Eq. 9: T_δγ(5.2) = 1998 K'),
    ('T_γε at P = 0', T_gamma_epsilon(0), 575, 'Eq. 7: T_γε(0) = 575 K'),
    ('T_αε at P = 7.3 GPa', T_alpha_epsilon(7.3*GPa), 820, 'Eq. 10: T_αε(7.3) = 820 K'),
    ('T_αε at P = 15.8 GPa', T_alpha_epsilon(15.8*GPa), 300, 'Eq. 10: T_αε(15.8) = 300 K'),
    ('T_melt at triple pt', T_melt_Fe(98.5*GPa), 3712, 'Eq. 6: T_m(98.5) = 3712 K'),
]

print(f"{'Test':30s} {'Calculated':>12s} {'Expected':>12s} {'Status':>8s}")
print("-" * 66)

all_passed = True
for name, calc, expected, note in reference_values:
    diff = abs(calc - expected)
    status = "✓" if diff < 1.0 else "✗"
    if diff >= 1.0:
        all_passed = False
    print(f"{name:30s} {calc:12.1f} {expected:12.1f} {status:>8s}")

print("\n" + ("✓ All reference values match!" if all_passed else "✗ Some values differ"))

---
## Phase Diagram Visualization <a name="phase-diagram-viz"></a>

Generate the complete iron phase diagram following BICEPS Figure 2.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7), dpi=300)

# Temperature range
T_range = np.logspace(np.log10(300), 5, 500)  # 300 K to 1e5 K

# Melting curve
P_melt = np.logspace(5, 14, 500)  # Pa
T_melt = np.array([T_melt_Fe(P) for P in P_melt])
ax.loglog(T_melt, P_melt/GPa, linewidth=2, label='Melting')

# γ-ε boundary
P_ge = np.logspace(5, np.log10(100*GPa), 300)
T_ge = np.array([T_gamma_epsilon(P) for P in P_ge])
mask = P_ge > 8.47e9
ax.loglog(T_ge[mask], P_ge[mask]/GPa, linewidth=2, label='γ-ε')

# α-γ boundary
P_ag = np.linspace(1e5, 7.3*GPa, 100)
T_ag = np.array([T_alpha_gamma(P) for P in P_ag])
ax.loglog(T_ag, P_ag/GPa, linewidth=2, label='α-γ')

# δ-γ boundary
P_dg = np.linspace(1e5, 5.2*GPa, 100)
T_dg = np.array([T_delta_gamma(P) for P in P_dg])
ax.loglog(T_dg, P_dg/GPa, linewidth=2, label='δ-γ')

# α-ε boundary
P_ae = np.linspace(7.3*GPa, 15.8*GPa, 50)
T_ae = np.array([T_alpha_epsilon(P) for P in P_ae])
ax.loglog(T_ae, P_ae/GPa, linewidth=2, label='α-ε')

# Triple point marker
ax.plot(3712, 98.5, 'ko', markersize=8, markerfacecolor='yellow',
        markeredgewidth=1.5, label='Triple point', zorder=5)

# Phase labels
ax.text(450,  0.003, 'α',      fontsize=14, fontweight='bold')
ax.text(1300, 0.003, 'γ',      fontsize=14, fontweight='bold')
ax.text(1600, 0.003, 'δ',      fontsize=14, fontweight='bold')
ax.text(1500, 1000,  'ε',      fontsize=14, fontweight='bold')
ax.text(8000, 1,     'Liquid', fontsize=14, fontweight='bold')

ax.set_xlabel('Temperature [K]', fontsize=12)
ax.set_ylabel('Pressure [GPa]', fontsize=12)
ax.set_xlim(300, 1e5)
ax.set_ylim(1e5/GPa, 1e14/GPa)  # 1e-4 to 1e5 GPa
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

---
## Triple Point Verification <a name="triple-point"></a>

Verify that phase boundaries converge at the γ-ε-liquid triple point (98.5 GPa, 3712 K).

In [ ]:
print("Triple point verification (γ-ε-liquid):")
print("=" * 80)
print("Expected: P = 98.5 GPa, T = 3712 K\n")

P_triple = 98.5 * GPa

# Calculate temperatures at triple point pressure
T_ge_at_triple = T_gamma_epsilon(P_triple)
T_melt_at_triple = T_melt_Fe(P_triple)

print(f"At P = 98.5 GPa:")
print(f"  T_γε = {T_ge_at_triple:.1f} K")
print(f"  T_melt = {T_melt_at_triple:.1f} K")
print(f"  Expected: 3712 K")

# Check convergence
diff_ge = abs(T_ge_at_triple - 3712)
diff_melt = abs(T_melt_at_triple - 3712)

print(f"\nConvergence check:")
print(f"  |T_γε - 3712| = {diff_ge:.1f} K {'✓' if diff_ge < 50 else '⚠'}")
print(f"  |T_melt - 3712| = {diff_melt:.1f} K {'✓' if diff_melt < 1 else '⚠'}")

---
## Phase Determination Tests <a name="phase-determination"></a>

Test the `get_iron_phase()` function across the P-T space.

In [ ]:
print("Testing get_iron_phase() at representative conditions:")
print("=" * 80)

test_cases = [
    # (P [GPa], T [K], expected_phase, description)
    (1e-4, 300, 'alpha', 'Ambient conditions'),
    (1e-4, 1500, 'gamma', 'Low P, intermediate T'),
    (1e-4, 1700, 'delta', 'Low P, high T (below melting)'),
    (1e-4, 2000, 'liquid', 'Low P, above melting'),
    (5, 500, 'alpha', 'Moderate P, low T'),
    (5, 1200, 'gamma', 'Moderate P, intermediate T'),
    (10, 700, 'epsilon', 'P > 7.3 GPa, T > T_αε'),
    (50, 2000, 'gamma', 'High P, below γ-ε'),
    (500, 2000, 'epsilon', 'High P, above γ-ε'),
    (100, 3500, 'epsilon', 'Near triple point, solid'),
    (100, 4000, 'liquid', 'Near triple point, liquid'),
    (200, 4000, 'epsilon', 'Deep mantle conditions'),
    (200, 5500, 'liquid', 'Deep mantle, above melting'),
    (330, 5000, 'epsilon', 'Inner core boundary conditions'),
    (360, 6000, 'epsilon', 'Earth inner core'),
    (500, 6000, 'epsilon', 'Super-Earth core'),
    (1000, 8000, 'epsilon', 'Super-Earth deep core'),
]

print(f"{'P (GPa)':>10s} {'T (K)':>8s} {'Expected':>10s} {'Got':>10s} {'Status':>8s} {'Description'}")
print("-" * 80)

all_passed = True
for P_gpa, T, expected, desc in test_cases:
    P = P_gpa * GPa
    phase = get_iron_phase(P, T)
    status = "✓" if phase == expected else "✗"
    if phase != expected:
        all_passed = False
    print(f"{P_gpa:10.4f} {T:8.0f} {expected:>10s} {phase:>10s} {status:>8s} {desc}")

print("\n" + ("✓ All phase determinations correct!" if all_passed else "✗ Some phases incorrect"))

In [ ]:
# Visualize phase determination on a P-T grid
print("Generating phase map on P-T grid...")

# Create P-T grid (log-spaced)
P_grid = np.logspace(5, 14, 1000)  # 1e5 to 1e14 Pa
T_grid = np.logspace(np.log10(300), 5, 1000)  # 300 K to 1e5 K
TT, PP = np.meshgrid(T_grid, P_grid)

# Map phases to integers for plotting
phase_map = {'alpha': 0, 'delta': 1, 'gamma': 2, 'epsilon': 3, 'liquid': 4}
phase_colors = ['C0', 'C1', 'C2', 'C3', 'C4']

# Determine phase at each grid point
phase_grid = np.zeros_like(PP)
for i in range(PP.shape[0]):
    for j in range(PP.shape[1]):
        phase = get_iron_phase(PP[i,j], TT[i,j])
        phase_grid[i,j] = phase_map[phase]

# Plot
fig, ax = plt.subplots(figsize=(8, 7), dpi=300)

from matplotlib.colors import ListedColormap
cmap = ListedColormap(phase_colors)

im = ax.pcolormesh(TT, PP/GPa, phase_grid, cmap=cmap, shading='auto', vmin=-0.5, vmax=4.5)

# Add phase boundaries

# Melting curve
P_melt = np.logspace(5, 14, 500)  # Pa
T_melt = np.array([T_melt_Fe(P) for P in P_melt])
ax.plot(T_melt, P_melt/GPa, linewidth=0.5, c='k', alpha=0.5)

# γ-ε boundary
P_ge = np.logspace(5, np.log10(100*GPa), 300)
T_ge = np.array([T_gamma_epsilon(P) for P in P_ge])
mask = P_ge > 8.47e9
ax.plot(T_ge[mask], P_ge[mask]/GPa, linewidth=0.5, c='k', alpha=0.5)

# α-γ boundary
P_ag = np.linspace(1e5, 7.3*GPa, 100)
T_ag = np.array([T_alpha_gamma(P) for P in P_ag])
ax.plot(T_ag, P_ag/GPa, linewidth=0.5, c='k', alpha=0.5)

# δ-γ boundary
P_dg = np.linspace(1e5, 5.2*GPa, 100)
T_dg = np.array([T_delta_gamma(P) for P in P_dg])
ax.plot(T_dg, P_dg/GPa, linewidth=0.5, c='k', alpha=0.5)

# α-ε boundary
P_ae = np.linspace(7.3*GPa, 15.8*GPa, 50)
T_ae = np.array([T_alpha_epsilon(P) for P in P_ae])
ax.plot(T_ae, P_ae/GPa, linewidth=0.5, c='k', alpha=0.5)

# Colorbar
cbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2, 3, 4])
cbar.ax.set_yticklabels(['α (bcc)', 'δ (bcc)', 'γ (fcc)', 'ε (hcp)', 'liquid'])

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Temperature [K]', fontsize=12)
ax.set_ylabel('Pressure [GPa]', fontsize=12)
ax.set_xlim(300, 1e5)
ax.set_ylim(1e5/GPa, 1e14/GPa)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print("✓ Phase map generated successfully")

---
## EoS Selection Integration <a name="eos-selection"></a>

Test the `get_iron_eos()` and `get_iron_eos_for_PT()` functions.

In [ ]:
print("Testing get_iron_eos() phase-to-EoS mapping:")
print("=" * 80)

phase_eos_map = [
    ('alpha', 'Dorogokupets17', 'bcc'),
    ('delta', 'Dorogokupets17', 'bcc'),
    ('gamma', 'Dorogokupets17', 'fcc'),
    ('epsilon', 'Miozzi20', None),
    ('liquid', 'Luo24', None),
]

print(f"{'Phase':>10s} {'Expected EoS':>20s} {'Got':>20s} {'Status':>8s}")
print("-" * 62)

all_passed = True
for phase, expected_class, expected_phase in phase_eos_map:
    eos = get_iron_eos(phase)
    class_name = type(eos).__name__
    status = "✓" if class_name == expected_class else "✗"
    if class_name != expected_class:
        all_passed = False
    print(f"{phase:>10s} {expected_class:>20s} {class_name:>20s} {status:>8s}")

print("\n" + ("✓ All EoS mappings correct!" if all_passed else "✗ Some mappings incorrect"))

In [ ]:
print("Testing get_iron_eos_for_PT() automatic EoS selection:")
print("=" * 80)

# Test conditions including Miozzi20/Hakim18 switch at 310 GPa
test_conditions = [
    (1e-4, 300, 'alpha', 'Dorogokupets17'),
    (50, 2000, 'gamma', 'Dorogokupets17'),
    (100, 4000, 'liquid', 'Luo24'),
    (150, 4000, 'epsilon', 'Miozzi20'),  # Below 310 GPa
    (310, 5000, 'epsilon', 'Hakim18'),  # At threshold
    (320, 5500, 'epsilon', 'Hakim18'),  # Above 310 GPa
    (500, 6000, 'epsilon', 'Hakim18'),  # Super-Earth
    (1000, 8000, 'epsilon', 'Hakim18'),  # Deep super-Earth
]

print(f"{'P (GPa)':>10s} {'T (K)':>8s} {'Phase':>10s} {'Expected':>15s} {'Got':>15s} {'Status':>8s}")
print("-" * 72)

all_passed = True
for P_gpa, T, expected_phase, expected_eos in test_conditions:
    P = P_gpa * GPa
    eos, phase = get_iron_eos_for_PT(P, T)
    eos_name = type(eos).__name__
    
    phase_ok = phase == expected_phase
    eos_ok = eos_name == expected_eos
    status = "✓" if (phase_ok and eos_ok) else "✗"
    if not (phase_ok and eos_ok):
        all_passed = False
    
    print(f"{P_gpa:10.1f} {T:8.0f} {phase:>10s} {expected_eos:>15s} {eos_name:>15s} {status:>8s}")

print("\n" + ("✓ All automatic selections correct!" if all_passed else "✗ Some selections incorrect"))

In [ ]:
# Test that EoS instances return valid densities
print("Testing density continuity across EoS switches:")
print("=" * 80)

# Test along an adiabat through different phases
test_path = [
    (1, 1500),   # gamma
    (100, 3000),  # epsilon (Miozzi20)
    (200, 4500),  # epsilon (Miozzi20)
    (234.4, 5000), # epsilon (Hakim18 threshold)
    (300, 5500),  # epsilon (Hakim18)
    (500, 6500),  # epsilon (Hakim18)
]

print(f"\n{'P (GPa)':>10s} {'T (K)':>8s} {'Phase':>10s} {'EoS':>15s} {'ρ (kg/m³)':>12s}")
print("-" * 60)

for P_gpa, T in test_path:
    P = P_gpa * GPa
    eos, phase = get_iron_eos_for_PT(P, T)
    rho = eos.density(P, T)
    print(f"{P_gpa:10.1f} {T:8.0f} {phase:>10s} {type(eos).__name__:>15s} {rho:12.1f}")

print("\n✓ All densities computed successfully")

---
## Runtime Benchmarking <a name="runtime-phase"></a>

Benchmark the performance of phase determination functions.

In [ ]:
print("Runtime benchmarking for phase diagram functions:")
print("=" * 80)

n_calls = 50000

# Random P-T points across the phase diagram
np.random.seed(42)
P_random = 10**np.random.uniform(5, 13, n_calls)
T_random = 10**np.random.uniform(np.log10(300), 5, n_calls)

# Benchmark phase boundary functions
print(f"\nPhase boundary functions ({n_calls:,} calls each):")
print("-" * 50)

funcs = [
    ('T_melt_Fe', T_melt_Fe),
    ('T_gamma_epsilon', T_gamma_epsilon),
    ('T_alpha_gamma', T_alpha_gamma),
    ('T_delta_gamma', T_delta_gamma),
    ('T_alpha_epsilon', T_alpha_epsilon),
]

for name, func in funcs:
    start = time.perf_counter()
    for P in P_random:
        _ = func(P)
    elapsed = time.perf_counter() - start
    per_call = elapsed / n_calls * 1e6  # microseconds
    print(f"  {name:20s}: {elapsed:.3f} s total, {per_call:.2f} μs/call")

In [ ]:
# Benchmark get_iron_phase
print(f"\nget_iron_phase() ({n_calls:,} calls):")
print("-" * 50)

start = time.perf_counter()
for P, T in zip(P_random, T_random):
    _ = get_iron_phase(P, T)
elapsed = time.perf_counter() - start
per_call = elapsed / n_calls * 1e6
print(f"  get_iron_phase:     {elapsed:.3f} s total, {per_call:.2f} μs/call")

# Benchmark get_iron_eos_for_PT (includes EoS instantiation)
print(f"\nget_iron_eos_for_PT() ({n_calls:,} calls):")
print("-" * 50)

start = time.perf_counter()
for P, T in zip(P_random, T_random):
    _ = get_iron_eos_for_PT(P, T)
elapsed = time.perf_counter() - start
per_call = elapsed / n_calls * 1e6
print(f"  get_iron_eos_for_PT: {elapsed:.3f} s total, {per_call:.2f} μs/call")

print("\n✓ Benchmarking complete")

In [ ]:
# Full integration benchmark: phase determination + density calculation
print(f"\nFull integration: phase + density ({n_calls:,} calls):")
print("-" * 50)

failures_by_phase = {'alpha': [], 'delta': [], 'gamma': [], 'epsilon': [], 'liquid': []}
start = time.perf_counter()
for P, T in zip(P_random, T_random):
    eos, phase = get_iron_eos_for_PT(P, T)
    try:
        _ = eos.density(P, T)
    except RuntimeError:
        failures_by_phase[phase].append((P, T))
elapsed = time.perf_counter() - start
per_call = elapsed / n_calls * 1e3  # milliseconds
calls_per_sec = n_calls / elapsed
n_failures = sum(len(pts) for pts in failures_by_phase.values())

print(f"  Total time:        {elapsed:.2f} s")
print(f"  Per call:          {per_call:.3f} ms")
print(f"  Throughput:        {calls_per_sec:.0f} calls/s")
print(f"  Non-convergences:  {n_failures} ({100*n_failures/n_calls:.2f}%)")
if n_failures > 0:
    print(f"    By phase:")
    for phase, pts in failures_by_phase.items():
        if len(pts) > 0:
            print(f"      {phase}: {len(pts)}")
            for P_gpa, T in pts[:5]:  # Show up to 5 examples
                print(f"        P = {P_gpa:.5e} Pa, T = {T:.0f} K")
            if len(pts) > 5:
                print(f"        ... and {len(pts) - 5} more")

print("\n✓ Full integration benchmark complete")

In [ ]:
# Plot nonconvergences

fig, ax = plt.subplots(figsize=(8, 7), dpi=300)

# Temperature range
T_range = np.logspace(np.log10(300), 5, 500)  # 300 K to 1e5 K

# Nonconvergences
markers = {'alpha': 'o', 'delta': 's', 'gamma': '^', 'epsilon': 'D', 'liquid': 'v'}
for phase, pts in failures_by_phase.items():
    if len(pts) > 0:
        T_fail = [pt[1] for pt in pts]
        P_fail = [pt[0] for pt in pts]  # Already in GPa
        ax.scatter(T_fail, np.array(P_fail)/GPa, marker=markers[phase], s=50, 
                   edgecolors='black', linewidths=0.5, label=f'{phase} ({len(pts)})', zorder=1)

# Melting curve
P_melt = np.logspace(5, 13, 500)  # Pa
T_melt = np.array([T_melt_Fe(P) for P in P_melt])
ax.loglog(T_melt, P_melt/GPa, linewidth=2)

# γ-ε boundary
P_ge = np.logspace(5, np.log10(100*GPa), 300)
T_ge = np.array([T_gamma_epsilon(P) for P in P_ge])
mask = P_ge > 8.47e9
ax.loglog(T_ge[mask], P_ge[mask]/GPa, linewidth=2)

# α-γ boundary
P_ag = np.linspace(1e5, 7.3*GPa, 100)
T_ag = np.array([T_alpha_gamma(P) for P in P_ag])
ax.loglog(T_ag, P_ag/GPa, linewidth=2)

# δ-γ boundary
P_dg = np.linspace(1e5, 5.2*GPa, 100)
T_dg = np.array([T_delta_gamma(P) for P in P_dg])
ax.loglog(T_dg, P_dg/GPa, linewidth=2)

# α-ε boundary
P_ae = np.linspace(7.3*GPa, 15.8*GPa, 50)
T_ae = np.array([T_alpha_epsilon(P) for P in P_ae])
ax.loglog(T_ae, P_ae/GPa, linewidth=2)

# Phase labels
ax.text(450,  0.003, 'α',      fontsize=14, fontweight='bold')
ax.text(1300, 0.003, 'γ',      fontsize=14, fontweight='bold')
ax.text(1600, 0.003, 'δ',      fontsize=14, fontweight='bold')
ax.text(1500, 1000,  'ε',      fontsize=14, fontweight='bold')
ax.text(8000, 1,     'Liquid', fontsize=14, fontweight='bold')

ax.set_xlabel('Temperature [K]', fontsize=12)
ax.set_ylabel('Pressure [GPa]', fontsize=12)
ax.set_xlim(300, 1e5)
ax.set_ylim(1e5/GPa, 1e13/GPa)  # 1e-4 to 1e5 GPa
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

## Reference Point Determination <a name="ref-point"></a>

Determine $U_0$ and $S_0$ Shifts for Triple Point Reference

Reference point: bcc-fcc-liquid triple point at $(P_0, T_0)$ = (5.2 GPa, 1991 K)
 
Goal: Compute the shifts needed so that $U(P_0, T_0)$ = 0 and $S(P_0, T_0)$ = 0 for all EoS classes.

In [ ]:
from iron_eos import ATOMIC_MASS_FE
from scipy.optimize import brentq

# Reference point: bcc-fcc-liquid triple point
P0 = 5.2e9   # Pa
T0 = 1991.0  # K

print("=" * 70)
print("Reference Point Shifts for U₀ and S₀")
print("=" * 70)
print(f"Reference point: P₀ = {P0/1e9:.1f} GPa, T₀ = {T0:.1f} K")
print(f"(bcc-fcc-liquid triple point)")
print("=" * 70)

# Dictionary to store results
results = {}

# -----------------------------------------------------------------------------
# Solid phases at the triple point (bcc, fcc, Miozzi20)
# -----------------------------------------------------------------------------
solid_eos_at_triple = [
    ("Dorogokupets17 (bcc)", Dorogokupets17(phase='bcc')),
    ("Dorogokupets17 (fcc)", Dorogokupets17(phase='fcc')),
    ("Miozzi20 (hcp)", Miozzi20()),
]

print("\n--- Solid Phases at Triple Point ---\n")
print(f"{'EoS Class':<25} {'U [MJ/kg]':>12} {'S [kJ/(kg·K)]':>15} {'U₀ shift [MJ/kg]':>18} \
{'S₀ shift [kJ/(kg·K)]':>22}")
print("-" * 95)

for name, eos in solid_eos_at_triple:
    try:
        U = eos.specific_internal_energy(P0, T0)
        S = eos.specific_entropy(P0, T0)
        
        # Required shifts to make U=0 and S=0 at reference
        U0_shift = -U
        S0_shift = -S
        
        results[name] = {
            'U': U,
            'S': S,
            'U0_shift': U0_shift,
            'S0_shift': S0_shift,
            'P_ref': P0,
            'T_ref': T0
        }
        
        print(f"{name:<25} {U/1e6:>12.4f} {S/1e3:>15.4f} {U0_shift/1e6:>18.4f} {S0_shift/1e3:>22.4f}")
        
    except Exception as e:
        print(f"{name:<25} {'FAILED':>12} - {str(e)[:50]}")
        results[name] = {'error': str(e)}

# -----------------------------------------------------------------------------
# Hakim18: Evaluated at 310 GPa (Miozzi20/Hakim18 transition pressure)
# We use the γ-ε phase boundary temperature at this pressure
# -----------------------------------------------------------------------------
print("\n--- Hakim18 (high-P hcp) ---")
print("Note: Hakim18 is valid at P > ~230 GPa. Evaluating at the transition point.\n")

P_hakim_ref = 310e9  # Pa - transition pressure to Hakim18
T_hakim_ref = T_gamma_epsilon(P_hakim_ref)  # Temperature on γ-ε boundary

eos_hakim = Hakim18()
eos_miozzi = Miozzi20()

try:
    # Get values from both EoS at the transition point
    U_hakim = eos_hakim.specific_internal_energy(P_hakim_ref, T_hakim_ref)
    S_hakim = eos_hakim.specific_entropy(P_hakim_ref, T_hakim_ref)
    
    U_miozzi_at_trans = eos_miozzi.specific_internal_energy(P_hakim_ref, T_hakim_ref)
    S_miozzi_at_trans = eos_miozzi.specific_entropy(P_hakim_ref, T_hakim_ref)
    
    # For Hakim18, the shift should ensure continuity with Miozzi20 at 310 GPa
    # After applying Miozzi20's shift: U_miozzi = U_miozzi_raw + U0_shift_miozzi
    # We want U_hakim + U0_shift_hakim = U_miozzi_raw + U0_shift_miozzi
    # So: U0_shift_hakim = (U_miozzi_raw - U_hakim) + U0_shift_miozzi
    
    U0_miozzi_shift = results['Miozzi20 (hcp)']['U0_shift']
    S0_miozzi_shift = results['Miozzi20 (hcp)']['S0_shift']
    
    U0_hakim_shift = (U_miozzi_at_trans - U_hakim) + U0_miozzi_shift
    S0_hakim_shift = (S_miozzi_at_trans - S_hakim) + S0_miozzi_shift
    
    results['Hakim18 (hcp)'] = {
        'U': U_hakim,
        'S': S_hakim,
        'U0_shift': U0_hakim_shift,
        'S0_shift': S0_hakim_shift,
        'P_ref': P_hakim_ref,
        'T_ref': T_hakim_ref,
        'note': 'Shift ensures continuity with Miozzi20 at 310 GPa'
    }
    
    print(f"Evaluation point: P = {P_hakim_ref/1e9:.0f} GPa, T = {T_hakim_ref:.0f} K (γ-ε boundary)")
    print(f"  Hakim18:  U = {U_hakim/1e6:.4f} MJ/kg,  S = {S_hakim/1e3:.4f} kJ/(kg·K)")
    print(f"  Miozzi20: U = {U_miozzi_at_trans/1e6:.4f} MJ/kg,  S = {S_miozzi_at_trans/1e3:.4f} kJ/(kg·K)")
    print(f"  Jump ΔU = {(U_hakim - U_miozzi_at_trans)/1e6:.4f} MJ/kg, ΔS = \
    {(S_hakim - S_miozzi_at_trans)/1e3:.4f} kJ/(kg·K)")
    print(f"\n  Required shifts for Hakim18 (to match shifted Miozzi20):")
    print(f"  U₀ = {U0_hakim_shift/1e6:.4f} MJ/kg,  S₀ = {S0_hakim_shift/1e3:.4f} kJ/(kg·K)")
    
except Exception as e:
    print(f"FAILED: {e}")
    results['Hakim18 (hcp)'] = {'error': str(e)}

# -----------------------------------------------------------------------------
# Liquid phases - shifts for continuity at melting curve
# -----------------------------------------------------------------------------
print("\n--- Liquid Phases ---")
print("Note: Liquid EoS have limited T ranges. Computing shifts for continuity")
print("      with solid phase (fcc or hcp) at the melting curve.\n")

# For liquid, we want continuity with the solid phase just below the melting curve
# At low P: liquid ↔ fcc (γ)
# At high P: liquid ↔ hcp (ε)

liquid_eos_configs = [
    ("Luo24 (liquid)", Luo24(), 6500.0),          # Converges above ~6000 K
    ("Ichikawa20 (liquid)", Ichikawa20(), 7500.0), # Better at intermediate T
]

for name, eos_liq, T_eval in liquid_eos_configs:
    try:
        # Find pressure on melting curve at T_eval
        P_melt = brentq(lambda P: T_melt_Fe(P) - T_eval, 5e9, 800e9)
        
        # At this P, the solid phase is hcp (ε) since P > 98.5 GPa (ε-γ-liquid triple point)
        # Use Miozzi20 for P < 310 GPa, Hakim18 for P > 310 GPa
        if P_melt < 310e9:
            eos_solid = Miozzi20()
            solid_name = "Miozzi20"
            U0_solid_shift = results['Miozzi20 (hcp)']['U0_shift']
            S0_solid_shift = results['Miozzi20 (hcp)']['S0_shift']
        else:
            eos_solid = Hakim18()
            solid_name = "Hakim18"
            U0_solid_shift = results['Hakim18 (hcp)']['U0_shift']
            S0_solid_shift = results['Hakim18 (hcp)']['S0_shift']
        
        # Evaluate both EoS just at the melting temperature
        U_liq = eos_liq.specific_internal_energy(P_melt, T_eval)
        S_liq = eos_liq.specific_entropy(P_melt, T_eval)
        
        U_solid = eos_solid.specific_internal_energy(P_melt, T_eval)
        S_solid = eos_solid.specific_entropy(P_melt, T_eval)
        
        # Shift for liquid to match shifted solid at melting point
        # This ignores latent heat - for consistency we match values
        # (In reality, S_liquid > S_solid by ΔS_fusion)
        U0_liq_shift = (U_solid - U_liq) + U0_solid_shift
        S0_liq_shift = (S_solid - S_liq) + S0_solid_shift
        
        results[name] = {
            'U': U_liq,
            'S': S_liq,
            'U0_shift': U0_liq_shift,
            'S0_shift': S0_liq_shift,
            'P_ref': P_melt,
            'T_ref': T_eval,
            'note': f'Shift ensures continuity with {solid_name} at melting curve'
        }
        
        print(f"{name}:")
        print(f"  Evaluation point: P = {P_melt/1e9:.1f} GPa, T = {T_eval:.0f} K (on melting curve)")
        print(f"  Solid reference: {solid_name}")
        print(f"  Liquid: U = {U_liq/1e6:.4f} MJ/kg,  S = {S_liq/1e3:.4f} kJ/(kg·K)")
        print(f"  Solid:  U = {U_solid/1e6:.4f} MJ/kg,  S = {S_solid/1e3:.4f} kJ/(kg·K)")
        print(f"  Raw jump: ΔU = {(U_liq - U_solid)/1e6:.4f} MJ/kg, ΔS = {(S_liq - S_solid)/1e3:.4f} kJ/(kg·K)")
        print(f"  Required shifts (for continuity, ignoring latent heat):")
        print(f"    U₀ = {U0_liq_shift/1e6:.4f} MJ/kg,  S₀ = {S0_liq_shift/1e3:.4f} kJ/(kg·K)")
        print()
        
    except Exception as e:
        print(f"{name}: FAILED - {str(e)[:60]}")
        results[name] = {'error': str(e)}

# -----------------------------------------------------------------------------
# Summary: Molar values for implementation
# -----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("Summary: Molar Shifts for Implementation (J/mol and J/(mol·K))")
print("=" * 70)
print(f"\n{'EoS Class':<25} {'U₀ [kJ/mol]':>14} {'S₀ [J/(mol·K)]':>16}")
print("-" * 58)

for name, data in results.items():
    if 'error' not in data:
        # Convert from specific (J/kg) to molar (J/mol)
        U0_molar = data['U0_shift'] * ATOMIC_MASS_FE
        S0_molar = data['S0_shift'] * ATOMIC_MASS_FE
        print(f"{name:<25} {U0_molar/1e3:>14.3f} {S0_molar:>16.3f}")

In [ ]:
# Check that U(P₀, T₀) ≈ 0 and S(P₀, T₀) ≈ 0 for all EoS classes.

# Reference point
P0, T0 = 5.2e9, 1991.0  # Pa, K

eos_classes = [
    ("Dorogokupets17 (bcc)", Dorogokupets17(phase='bcc')),
    ("Dorogokupets17 (fcc)", Dorogokupets17(phase='fcc')),
    ("Miozzi20 (hcp)", Miozzi20()),
    ("Hakim18 (hcp)", Hakim18()),
    ("Luo24 (liquid)", Luo24()),
    ("Ichikawa20 (liquid)", Ichikawa20()),
]

print(f"Reference point: P₀ = {P0/1e9:.1f} GPa, T₀ = {T0:.0f} K\n")
print(f"{'EoS Class':<25} {'U [J/kg]':>15} {'S [J/(kg·K)]':>15} {'Status':>10}")
print("-" * 68)

all_ok = True
for name, eos in eos_classes:
    try:
        U = eos.specific_internal_energy(P0, T0)
        S = eos.specific_entropy(P0, T0)
        ok = abs(U) < 1e3 and abs(S) < 1.0  # Tolerances: 1 kJ/kg, 1 J/(kg·K)
        status = "✓" if ok else "✗"
        if not ok:
            all_ok = False
        print(f"{name:<25} {U:>15.2f} {S:>15.4f} {status:>10}")
    except Exception as e:
        print(f"{name:<25} {'N/A':>15} {'N/A':>15} {'(no conv)':>10}")

print("-" * 68)
print("All checks passed!" if all_ok else "Some values are not zero—check shifts.")

---
## Thermodynamic Properties Maps <a name="thermo-maps"></a>

Generate and visualize all thermodynamic quantities across the P-T space using phase-appropriate EoS.

In [ ]:
# Generate thermodynamic properties on P-T grid
print("Generating thermodynamic properties on P-T grid...")
print("=" * 80)

# Create log-spaced P-T grid
P_grid = np.logspace(5, 13, 200)  # 1e5 to 1e13 Pa
T_grid = np.logspace(np.log10(300), 5, 200)  # 300 K to 1e5 K
TT, PP = np.meshgrid(T_grid, P_grid)

# Properties to compute
properties = [
    ('density', 'kg/m³'),
    ('specific_internal_energy', 'J/kg'),
    ('specific_entropy', 'J/(kg·K)'),
    ('isobaric_heat_capacity', 'J/(kg·K)'),
    ('isochoric_heat_capacity', 'J/(kg·K)'),
    ('thermal_expansion', 'K⁻¹'),
    ('adiabatic_gradient', ''),
]

# Initialize storage
prop_grids = {name: np.full_like(PP, np.nan) for name, _ in properties}
phase_grid = np.empty(PP.shape, dtype=object)
n_failures = 0

# Compute properties at each grid point
for i in range(PP.shape[0]):
    for j in range(PP.shape[1]):
        P, T = PP[i, j], TT[i, j]
        eos, phase = get_iron_eos_for_PT(P, T)
        phase_grid[i, j] = phase
        
        for prop_name, _ in properties:
            try:
                method = getattr(eos, prop_name)
                prop_grids[prop_name][i, j] = method(P, T)
            except RuntimeError:
                n_failures += 1

print(f"✓ Grid computed: {PP.shape[0]} × {PP.shape[1]} = {PP.size:,} points")
print(f"  Non-convergences: {n_failures} ({100*n_failures/PP.size/len(properties):.2f}%)")

In [ ]:
# Helper function to plot property map with phase boundaries
def plot_property_map(prop_name, units, prop_grid, vmin=None, vmax=None, log_scale=True):
    fig, ax = plt.subplots(figsize=(10, 7), dpi=300)
    
    # Plot property
    if log_scale:
        data = np.log10(np.abs(prop_grid))
        label = f'log₁₀ |{prop_name}|' + (f' [{units}]' if units else '')
    else:
        data = prop_grid
        label = prop_name + (f' [{units}]' if units else '')
    
    im = ax.pcolormesh(TT, PP/GPa, data, vmin=vmin, vmax=vmax, shading='auto', cmap='inferno')
    cbar = plt.colorbar(im, ax=ax, label=label)
    
    # Overlay phase boundaries
    
    # Melting curve
    P_melt = np.logspace(5, 13, 500)  # Pa
    T_melt = np.array([T_melt_Fe(P) for P in P_melt])
    ax.plot(T_melt, P_melt/GPa, 'w-', linewidth=1.5)

    # γ-ε boundary
    P_ge = np.logspace(5, np.log10(100*GPa), 300)
    T_ge = np.array([T_gamma_epsilon(P) for P in P_ge])
    mask = P_ge > 8.47e9
    ax.plot(T_ge[mask], P_ge[mask]/GPa, 'w-', linewidth=1.5)

    # α-γ boundary
    P_ag = np.linspace(1e5, 7.3*GPa, 100)
    T_ag = np.array([T_alpha_gamma(P) for P in P_ag])
    ax.plot(T_ag, P_ag/GPa, 'w-', linewidth=1.5)

    # δ-γ boundary
    P_dg = np.linspace(1e5, 5.2*GPa, 100)
    T_dg = np.array([T_delta_gamma(P) for P in P_dg])
    ax.plot(T_dg, P_dg/GPa, 'w-', linewidth=1.5)

    # α-ε boundary
    P_ae = np.linspace(7.3*GPa, 15.8*GPa, 50)
    T_ae = np.array([T_alpha_epsilon(P) for P in P_ae])
    ax.plot(T_ae, P_ae/GPa, 'w-', linewidth=1.5)
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Temperature [K]', fontsize=12)
    ax.set_ylabel('Pressure [GPa]', fontsize=12)
    ax.set_xlim(300, 1e5)
    ax.set_ylim(1e5/GPa, 1e13/GPa)
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_property_map('$\\rho$', 'kg/m³', prop_grids['density'], log_scale=True)

In [ ]:
plot_property_map('$U$', 'J/kg', prop_grids['specific_internal_energy'], log_scale=False, vmax=1e7)

In [ ]:
plot_property_map('$S$', 'J/(kg·K)', prop_grids['specific_entropy'], log_scale=False)

In [ ]:
plot_property_map('$C_P$', 'J/(kg·K)', prop_grids['isobaric_heat_capacity'], log_scale=False)

In [ ]:
plot_property_map('$C_V$', 'J/(kg·K)', prop_grids['isochoric_heat_capacity'], log_scale=False)

In [ ]:
plot_property_map('$\\alpha$', 'K⁻¹', prop_grids['thermal_expansion'], log_scale=True)

In [ ]:
plot_property_map('$\\nabla_\mathrm{ad}$', '', prop_grids['adiabatic_gradient'], log_scale=True)

In [ ]:
# Compute thermodynamic consistency parameter
# Delta = 1 - rho^2 (dS/dP)_T / (drho/dT)_P
# For a thermodynamically consistent EoS, Delta = 0 (Maxwell relation)

Delta = np.full_like(PP, np.nan)
phase_map = np.empty(PP.shape, dtype=object)

# Finite difference steps (relative)
dP_rel = 1e-4
dT_rel = 1e-4

for i in range(PP.shape[0]):
    for j in range(PP.shape[1]):
        P, T = PP[i, j], TT[i, j]
        
        try:
            eos, phase = get_iron_eos_for_PT(P, T)
            phase_map[i, j] = phase
            
            # Central differences for derivatives
            dP = P * dP_rel
            dT = T * dT_rel
            
            # (dS/dP)_T
            S_plus = eos.specific_entropy(P + dP, T)
            S_minus = eos.specific_entropy(P - dP, T)
            dS_dP = (S_plus - S_minus) / (2 * dP)
            
            # (drho/dT)_P
            rho_plus = eos.density(P, T + dT)
            rho_minus = eos.density(P, T - dT)
            drho_dT = (rho_plus - rho_minus) / (2 * dT)
            
            # Density at current point
            rho = eos.density(P, T)
            
            # Consistency parameter
            if abs(drho_dT) > 1e-20:
                Delta[i, j] = 1 - rho**2 * dS_dP / drho_dT
            
        except Exception:
            phase_map[i, j] = None
            continue

In [ ]:
plot_property_map('$\\Delta$', '', Delta, log_scale=True)

In [ ]:
# Statistics on thermodynamic consistency parameter Delta

print("=" * 70)
print("Thermodynamic Consistency: Δ = 1 - ρ² (∂S/∂P)_T / (∂ρ/∂T)_P")
print("Perfect consistency: Δ = 0")
print("=" * 70)

# Overall statistics
valid_mask = ~np.isnan(Delta)
Delta_valid = Delta[valid_mask]

print(f"\n{'Overall Statistics':^70}")
print("-" * 70)
print(f"Total grid points:    {Delta.size:>10}")
print(f"Valid points:         {valid_mask.sum():>10} ({100*valid_mask.sum()/Delta.size:.1f}%)")
print(f"Invalid/failed:       {(~valid_mask).sum():>10}")

print(f"\n{'Statistic':<20} {'Value':>15} {'|Δ|':>15}")
print("-" * 52)
print(f"{'Mean':<20} {np.mean(Delta_valid):>15.2e} {np.mean(np.abs(Delta_valid)):>15.2e}")
print(f"{'Median':<20} {np.median(Delta_valid):>15.2e} {np.median(np.abs(Delta_valid)):>15.2e}")
print(f"{'Std':<20} {np.std(Delta_valid):>15.2e} {np.std(np.abs(Delta_valid)):>15.2e}")
print(f"{'Min':<20} {np.min(Delta_valid):>15.2e}")
print(f"{'Max':<20} {np.max(Delta_valid):>15.2e}")

# Percentiles
print(f"\n{'Percentiles of |Δ|':^70}")
print("-" * 70)
percentiles = [50, 90, 95, 99, 99.9]
abs_Delta = np.abs(Delta_valid)
for p in percentiles:
    val = np.percentile(abs_Delta, p)
    print(f"  {p:>5.1f}%:  |Δ| ≤ {val:.2e}")

# Consistency thresholds
print(f"\n{'Consistency Assessment':^70}")
print("-" * 70)
thresholds = [1e-10, 1e-8, 1e-6, 1e-4, 1e-2, 0.1, 1.0]
for thresh in thresholds:
    count = np.sum(abs_Delta <= thresh)
    pct = 100 * count / len(abs_Delta)
    print(f"  |Δ| ≤ {thresh:<8.0e}:  {count:>8} points ({pct:>6.2f}%)")

# Per-phase statistics
print(f"\n{'Statistics by Phase':^70}")
print("-" * 70)
phases = ['alpha', 'delta', 'gamma', 'epsilon', 'liquid']
print(f"{'Phase':<12} {'Count':>8} {'Mean |Δ|':>12} {'Median |Δ|':>12} {'Max |Δ|':>12}")
print("-" * 60)

for phase in phases:
    mask = (phase_map == phase) & valid_mask
    if mask.sum() > 0:
        Delta_phase = Delta[mask]
        abs_Delta_phase = np.abs(Delta_phase)
        print(f"{phase:<12} {mask.sum():>8} {np.mean(abs_Delta_phase):>12.2e} "
              f"{np.median(abs_Delta_phase):>12.2e} {np.max(abs_Delta_phase):>12.2e}")
    else:
        print(f"{phase:<12} {0:>8} {'N/A':>12} {'N/A':>12} {'N/A':>12}")

# Flag problematic regions
print(f"\n{'Problematic Regions (|Δ| > 0.01)':^70}")
print("-" * 70)
problem_mask = np.abs(Delta) > 0.01
if problem_mask.sum() > 0:
    P_prob = PP[problem_mask]
    T_prob = TT[problem_mask]
    Delta_prob = Delta[problem_mask]
    phase_prob = phase_map[problem_mask]
    
    # Group by phase
    for phase in phases:
        phase_mask = phase_prob == phase
        if phase_mask.sum() > 0:
            P_range = (P_prob[phase_mask].min()/1e9, P_prob[phase_mask].max()/1e9)
            T_range = (T_prob[phase_mask].min(), T_prob[phase_mask].max())
            print(f"  {phase}: {phase_mask.sum()} points")
            print(f"    P: {P_range[0]:.1f} - {P_range[1]:.1f} GPa")
            print(f"    T: {T_range[0]:.0f} - {T_range[1]:.0f} K")
else:
    print("  None found - all points have |Δ| ≤ 0.01")

# Liquid Iron EoS: Convergence Troubleshooting

This section investigates non-convergence issues in the `find_volume` root-finding for liquid iron EoS:

- **Ichikawa20**: Non-convergence at $T > T_0 = 8000\,\mathrm{K}$
- **Luo24**: Non-convergence at $T < T_0 = 6000\,\mathrm{K}$

**Contents:**
1. Setup and initialization
2. Convergence maps in P-T space
3. P(V) - P_target isotherm analysis
4. Root cause diagnosis

---

## 1. Setup and Initialization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

from iron_eos import Ichikawa20, Luo24, T_melt_Fe

# Units
GPa = 1e9

# Initialize EoS instances
eos_ichikawa = Ichikawa20()
eos_luo = Luo24()

print("Liquid Iron EoS Troubleshooting")
print("=" * 60)
print(f"\nIchikawa20:")
print(f"  T₀ = {eos_ichikawa.T0:.0f} K (reference temperature)")
print(f"  V₀ = {eos_ichikawa.params['V0']*1e6:.4f} cm³/mol")
print(f"  K_T0 = {eos_ichikawa.params['KT0']/GPa:.2f} GPa")
print(f"  γ = {eos_ichikawa.params['gamma']:.2f}")
print(f"\nLuo24:")
print(f"  T₀ = {eos_luo.T0:.0f} K (reference temperature)")
print(f"  V₀ = {eos_luo.params['V0']*1e6:.4f} cm³/mol")
print(f"  K₀ = {eos_luo.params['K0']/GPa:.2f} GPa")

---
## 2. Convergence Maps in P-T Space

Visualize where `_find_volume` converges vs fails across the liquid P-T domain.

In [ ]:
def test_convergence_grid(eos, P_range, T_range, eos_name):
    """
    Test convergence across P-T grid and return convergence map + volume grid.
    
    Returns:
        converged: bool array (True = converged)
        V_grid: volume array (NaN where failed)
        PP, TT: meshgrid arrays
    """
    TT, PP = np.meshgrid(T_range, P_range)
    converged = np.zeros(PP.shape, dtype=bool)
    V_grid = np.full(PP.shape, np.nan)
    
    total = PP.size
    n_failed = 0
    
    for i in range(PP.shape[0]):
        for j in range(PP.shape[1]):
            P, T = PP[i, j], TT[i, j]
            try:
                V = eos._find_volume(P, T)
                converged[i, j] = True
                V_grid[i, j] = V
            except RuntimeError:
                n_failed += 1
    
    print(f"{eos_name}: {n_failed}/{total} non-convergences ({100*n_failed/total:.1f}%)")
    return converged, V_grid, PP, TT

In [ ]:
# Define P-T grids for liquid iron region
# Focus on regions around T0 = 8000 K and problematic zones

P_range = np.logspace(5, 13, 100)  # 0.1 MPa - 10 TPa
T_range = np.logspace(np.log10(2000), 5, 120)  # 2000 - 100,000 K

print("Testing convergence across P-T grid...")
print("=" * 60)

conv_ich, V_ich, PP, TT = test_convergence_grid(eos_ichikawa, P_range, T_range, "Ichikawa20")
conv_luo, V_luo, _, _ = test_convergence_grid(eos_luo, P_range, T_range, "Luo24")

In [ ]:
# Plot convergence maps side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=300)

# Custom colormap: green = converged, red = failed
cmap_conv = ListedColormap(['#d62728', '#2ca02c'])  # red, green

for ax, conv, V_grid, title in [(axes[0], conv_ich, V_ich, 'Ichikawa20'),
                                 (axes[1], conv_luo, V_luo, 'Luo24')]:
    
    # Plot convergence status
    im = ax.pcolormesh(TT, PP/GPa, conv.astype(int), cmap=cmap_conv, 
                       vmin=0, vmax=1, shading='auto', alpha=0.7)
    
    # Overlay melting curve for context
    P_melt_arr = np.logspace(5, 13, 200)
    T_melt_arr = np.array([T_melt_Fe(P) for P in P_melt_arr])
    ax.plot(T_melt_arr, P_melt_arr/GPa, 'k--', lw=2, label='Melting curve')
    
    # Mark T0 = 8000 K reference
    ax.axvline(x=8000, color='blue', ls=':', lw=1.5, label='$T_0 = 8000$ K')
    
    ax.set_xlabel('Temperature [K]', fontsize=12)
    ax.set_ylabel('Pressure [GPa]', fontsize=12)
    ax.loglog()
    ax.legend(loc='upper left', fontsize=9)
    ax.set_xlim([TT.min(), TT.max()])
    ax.set_ylim([(PP/GPa).min(), (PP/GPa).max()])
    
    # Custom legend for convergence
    legend_elements = [Patch(facecolor='#2ca02c', label='Converged'),
                       Patch(facecolor='#d62728', label='Failed')]
    ax.legend(handles=legend_elements + ax.get_legend_handles_labels()[0], 
              loc='upper left', fontsize=9)

axes[0].set_title('Ichikawa20', fontsize=12)
axes[1].set_title('Luo24', fontsize=12)

plt.tight_layout()
plt.show()

print("\n→ Ichikawa20 fails for T > T₀ (especially at lower pressures)")
print("→ Luo24 fails for T < ~6000 K (especially at higher pressures)")

In [ ]:
# Plot molar volume where converged
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=300)

for ax, V_grid, eos, title in [(axes[0], V_ich, eos_ichikawa, 'Ichikawa20'),
                                (axes[1], V_luo, eos_luo, 'Luo24')]:
    
    # Convert to cm³/mol for readability
    V_cm3 = V_grid * 1e6
    V0_cm3 = eos.params['V0'] * 1e6
    
    im = ax.pcolormesh(TT, PP/GPa, V_cm3, shading='auto', cmap='viridis', vmin=2, vmax=10)
    cbar = plt.colorbar(im, ax=ax, label='Molar Volume [cm³/mol]')
    
    # Overlay melting curve
    P_melt_arr = np.logspace(5, 13, 200)
    T_melt_arr = np.array([T_melt_Fe(P) for P in P_melt_arr])
    ax.plot(T_melt_arr, P_melt_arr/GPa, 'r--', lw=2, label='Melting curve')
    
    ax.axvline(x=8000, color='white', ls=':', lw=1.5, label='$T_0 = 8000$ K')
    
    ax.set_xlim([TT.min(), TT.max()])
    ax.set_ylim([(PP/GPa).min(), (PP/GPa).max()])
    
    ax.set_xlabel('Temperature [K]', fontsize=12)
    ax.set_ylabel('Pressure [GPa]', fontsize=12)

    ax.loglog()
    ax.legend(loc='upper left', fontsize=9)

axes[0].set_title('Ichikawa20', fontsize=12)
axes[1].set_title('Luo24', fontsize=12)
    
plt.tight_layout()
plt.show()

---
## 3. P(V) - P_target Isotherm Analysis

Plot the residual function `P(V,T) - P_target` for several isotherms to visualize why Brent's method fails to find roots.

In [ ]:
def plot_pressure_residuals(eos, eos_name, T_values, P_targets, V_bounds_factor=(0.3, 2.0)):
    """
    Plot P(V,T) - P_target for multiple isotherms and target pressures.
    
    Parameters:
        eos: EoS instance
        eos_name: str for title
        T_values: list of temperatures to plot
        P_targets: list of target pressures [Pa]
        V_bounds_factor: (min, max) factors of V0 for volume range
    """
    V0 = eos.params['V0']
    V_min = V_bounds_factor[0] * V0
    V_max = V_bounds_factor[1] * V0
    V_range = np.linspace(V_min, V_max, 500)
    
    n_targets = len(P_targets)
    fig, axes = plt.subplots(1, n_targets, figsize=(6*n_targets, 5), dpi=150, squeeze=False)
    axes = axes[0]
    
    colors = plt.cm.coolwarm(np.linspace(0, 1, len(T_values)))
    
    for ax, P_target in zip(axes, P_targets):
        ax.axhline(y=0, color='black', lw=1.5, ls='-', zorder=10)
        
        for T, color in zip(T_values, colors):
            # Compute P(V) at this isotherm
            P_of_V = np.array([eos._total_pressure(V, T) for V in V_range])
            residual = P_of_V - P_target
            
            # Check if root exists in interval
            has_root = (residual[0] * residual[-1]) < 0
            ls = '-' if has_root else '--'
            lw = 2 if has_root else 1
            
            label = f'$T = {T:.0f}$ K' + ('' if has_root else ' (no root)')
            ax.plot(V_range * 1e6, residual / GPa, color=color, ls=ls, lw=lw, label=label)
        
        ax.set_xlabel('$V$ [cm³/mol]', fontsize=12)
        ax.set_ylabel('$P(V,T) - P_\\mathrm{target}$ [GPa]', fontsize=12)
        ax.set_title(f'{eos_name}: $P_\\mathrm{{target}} = {P_target/GPa:.0f}$ GPa', fontsize=13)
        ax.legend(loc='best', fontsize=8, ncol=1)
        ax.grid(True, alpha=0.3)
        
        # Mark V0
        #ax.axvline(x=V0*1e6, color='gray', ls=':', lw=1, alpha=0.7)
        #ax.text(V0*1e6, ax.get_ylim()[1]*0.9, '$V_0$', fontsize=10, ha='center')
    
    plt.tight_layout()
    plt.show()

### 3.1 Ichikawa20: Analysis for T > T₀ = 8000 K

For $T > T_0$, thermal pressure is positive, raising the minimum achievable pressure.

In [ ]:
# Ichikawa20: Focus on temperatures above and below T0 = 8000 K
T_values_ich = [5000, 6000, 7000, 8000, 9000, 10000, 11000, 12000]
P_targets_ich = [1*GPa, 1e-3*GPa]

print("Ichikawa20: P(V) - P_target residuals")
print("Solid lines = root exists, Dashed lines = no root in [V_min, V_max]")
print("=" * 70)

plot_pressure_residuals(eos_ichikawa, 'Ichikawa20', T_values_ich, P_targets_ich, 
                        V_bounds_factor=(5/11, 3))

In [ ]:
# Diagnostic: What is the minimum pressure achievable at each temperature?
def compute_pressure_bounds(eos, T_values, V_bounds_factor=(0.3, 2.0)):
    """
    Compute min and max pressure achievable at each temperature within volume bounds.
    """
    V0 = eos.params['V0']
    V_min = V_bounds_factor[0] * V0
    V_max = V_bounds_factor[1] * V0
    
    results = []
    for T in T_values:
        P_at_Vmin = eos._total_pressure(V_min, T)
        P_at_Vmax = eos._total_pressure(V_max, T)
        P_min = min(P_at_Vmin, P_at_Vmax)
        P_max = max(P_at_Vmin, P_at_Vmax)
        results.append({'T': T, 'P_min': P_min, 'P_max': P_max})
    return results

print("\nIchikawa20: Pressure bounds at V ∈ [0.3V₀, 2.0V₀]")
print("=" * 60)
print(f"{'T [K]':>10} {'P_min [GPa]':>15} {'P_max [GPa]':>15}")
print("-" * 40)

T_diagnostic = np.arange(5000, 13000, 500)
bounds_ich = compute_pressure_bounds(eos_ichikawa, T_diagnostic)
for b in bounds_ich:
    marker = " ← T₀" if b['T'] == 8000 else ""
    print(f"{b['T']:>10.0f} {b['P_min']/GPa:>15.1f} {b['P_max']/GPa:>15.1f}{marker}")

### 3.2 Luo24: Analysis for T < 6000 K

For $T < T_0 = 6000\,\mathrm{K}$, two solutions exist for the volume.

In [ ]:
# Luo24: Focus on temperatures below T0
T_values_luo = [2000, 3000, 4000, 5000, 5500, 6000, 6500, 7000]
P_targets_luo = [1*GPa, 1e-3*GPa]

print("Luo24: P(V) - P_target residuals")
print("Solid lines = root exists, Dashed lines = no root in [V_min, V_max]")
print("=" * 70)

plot_pressure_residuals(eos_luo, 'Luo24', T_values_luo, P_targets_luo,
                        V_bounds_factor=(0.5, 1.5))

In [ ]:
Vb_min = 0.3
Vb_max = 2.

print(f"\nLuo24: Pressure bounds at V ∈ [{Vb_min} V₀, {Vb_max} V₀]")
print("=" * 60)
print(f"{'T [K]':>10} {'P_min [GPa]':>15} {'P_max [GPa]':>15}")
print("-" * 40)

T_diagnostic_luo = np.arange(2000, 10000, 500)
bounds_luo = compute_pressure_bounds(eos_luo, T_diagnostic_luo, V_bounds_factor=(Vb_min, Vb_max))
for b in bounds_luo:
    marker = " ← T₀" if b['T'] == 8000 else ""
    print(f"{b['T']:>10.0f} {b['P_min']/GPa:>15.1f} {b['P_max']/GPa:>15.1f}{marker}")

print("\n→ Note: P_max decreases dramatically for T < 6000 K")
print("→ This is why high target pressures have no solution at low T")

---
## 4. Root Cause Diagnosis

Examine the thermal pressure contribution directly to understand the physics.

In [ ]:
def plot_pressure_components(eos, eos_name, T_values):
    """
    Plot P_cold(V), P_th(V,T), and P_total(V,T) for multiple isotherms.
    """
    V0 = eos.params['V0']
    V_range = np.linspace(0.3*V0, 2.*V0, 300)
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=150)
    colors = plt.cm.coolwarm(np.linspace(0, 1, len(T_values)))
    
    # Cold pressure (same for all T)
    P_cold = np.array([eos._cold_pressure(V) for V in V_range])
    axes[0].plot(V_range*1e6, P_cold/GPa, 'k-', lw=2, label='$P_\\mathrm{cold}$ (at $T_0$)')
    axes[0].set_xlabel('$V$ [cm³/mol]', fontsize=12)
    axes[0].set_ylabel('$P_\\mathrm{cold}$ [GPa]', fontsize=12)
    axes[0].set_title(f'{eos_name}: Cold Pressure', fontsize=13, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].axvline(x=V0*1e6, color='gray', ls=':', alpha=0.5)
    
    # Thermal pressure for each T
    for T, color in zip(T_values, colors):
        P_th = np.array([eos._thermal_pressure(V, T) for V in V_range])
        P_total = np.array([eos._total_pressure(V, T) for V in V_range])
        
        label = f'$T = {T:.0f}$ K'
        axes[1].plot(V_range*1e6, P_th/GPa, color=color, lw=1.5, label=label)
        axes[2].plot(V_range*1e6, P_total/GPa, color=color, lw=1.5, label=label)
    
    axes[1].axhline(y=0, color='black', lw=1, ls='-')
    axes[1].set_xlabel('$V$ [cm³/mol]', fontsize=12)
    axes[1].set_ylabel('$P_\\mathrm{th}$ [GPa]', fontsize=12)
    axes[1].set_title(f'{eos_name}: Thermal Pressure', fontsize=13, fontweight='bold')
    axes[1].legend(loc='best', fontsize=8)
    axes[1].grid(True, alpha=0.3)
    axes[1].axvline(x=V0*1e6, color='gray', ls=':', alpha=0.5)
    
    axes[2].set_xlabel('$V$ [cm³/mol]', fontsize=12)
    axes[2].set_ylabel('$P$ [GPa]', fontsize=12)
    axes[2].set_title(f'{eos_name}: Total Pressure', fontsize=13, fontweight='bold')
    axes[2].legend(loc='best', fontsize=8)
    axes[2].grid(True, alpha=0.3)
    axes[2].axvline(x=V0*1e6, color='gray', ls=':', alpha=0.5)
    axes[2].semilogy()
    
    plt.tight_layout()
    plt.show()

In [ ]:
print("Ichikawa20: Pressure Components")
print("=" * 60)
T_values_ich_diag = [5000, 6000, 7000, 8000, 9000, 10000, 11000, 12000]
plot_pressure_components(eos_ichikawa, 'Ichikawa20', T_values_ich_diag)

In [ ]:
print("Luo24: Pressure Components")
print("=" * 60)
T_values_luo_diag = [2000, 3000, 4000, 5000, 5500, 6000, 6500, 7000]
plot_pressure_components(eos_luo, 'Luo24', T_values_luo_diag)

In [ ]:
def compute_valid_domain(eos, T_range, V_bounds_factor=(0.3, 2.0)):
    """
    For each temperature, compute the min and max pressure that can be solved.
    """
    V0 = eos.params['V0']
    V_min = V_bounds_factor[0] * V0
    V_max = V_bounds_factor[1] * V0
    
    P_min_arr = []
    P_max_arr = []
    
    for T in T_range:
        # Sample many volumes to find min/max pressure
        V_sample = np.linspace(V_min, V_max, 200)
        P_sample = [eos._total_pressure(V, T) for V in V_sample]
        P_min_arr.append(min(P_sample))
        P_max_arr.append(max(P_sample))
    
    return np.array(P_min_arr), np.array(P_max_arr)

# Compute valid domains
T_domain = np.logspace(np.log10(2000), 5, 200)

P_min_ich, P_max_ich = compute_valid_domain(eos_ichikawa, T_domain, V_bounds_factor=(0.1, 10.))
P_min_luo, P_max_luo = compute_valid_domain(eos_luo, T_domain, V_bounds_factor=(0.1, 10.))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=150)

for ax, P_min, P_max, eos_name in [(axes[0], P_min_ich, P_max_ich, 'Ichikawa20'),
                                    (axes[1], P_min_luo, P_max_luo, 'Luo24')]:
    # Fill valid region
    ax.fill_between(T_domain, P_min/GPa, P_max/GPa, alpha=0.3, color='green', label='Valid domain')
    ax.plot(T_domain, P_min/GPa, 'g-', lw=2)
    ax.plot(T_domain, P_max/GPa, 'g-', lw=2)
    
    # Melting curve for reference
    P_melt_arr = np.logspace(5, 13, 200)
    T_melt_arr = np.array([T_melt_Fe(P) for P in P_melt_arr])
    ax.plot(T_melt_arr, P_melt_arr/GPa, 'k--', lw=2, label='Melting curve')
    
    ax.axvline(x=8000, color='blue', ls=':', lw=1.5, label='$T_0 = 8000$ K')
    
    ax.set_xlabel('Temperature [K]', fontsize=12)
    ax.set_ylabel('Pressure [GPa]', fontsize=12)
    ax.loglog()
    ax.set_ylim(1e-4, 1e4)
    ax.set_xlim(2000, 1e5)
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True, alpha=0.3)

axes[0].set_title('Ichikawa20', fontsize=12)
axes[1].set_title('Luo24', fontsize=12)
    
plt.tight_layout()
plt.show()